# Sentiment Distillation — Kaggle / Colab

Notebook hỗ trợ clone repo private bằng GitHub token hoặc sử dụng source đã đóng gói.
Upload duy nhất file `.ipynb` này lên Kaggle, bật **GPU** và **Internet**, chạy lần lượt các cell.
Colab: chọn runtime GPU. Đặt `USE_PRIVATE_REPO=False` nếu muốn dùng bản đóng gói, không cần GitHub token.

Mặc định: Qwen 1.5B, SST2, 32 mẫu đầu, batch 1, 4-shot, 2.048 token đầu vào.
Đây là chạy thử, không phải kết quả toàn benchmark. Không có huấn luyện/fine-tuning trong notebook.

Môi trường riêng dùng `--system-site-packages` để tái sử dụng PyTorch/CUDA của nền tảng.
Các gói dự án được cài trong venv; các chương trình đánh giá chạy bằng Python của venv,
không dùng kernel để import các thư viện đã thay phiên bản. Đây là cách ly một phần, không phải container.
Có kiểm tra kế hoạch pip, dependency đang dùng, import, tiny-model forward/generation và GPU.
Nếu không đạt, notebook dừng. Chưa được chạy thực tế trên Kaggle; kiểm tra sẽ thực hiện trong phiên của bạn.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, shutil

BASE = Path('/kaggle/working/sentiment-distillation' if Path('/kaggle').exists()
            else '/content/sentiment-distillation' if Path('/content').exists()
            else './sentiment-notebook-run').resolve()
BASE.mkdir(parents=True, exist_ok=True)
USE_PRIVATE_REPO = True
PRIVATE_REPO = 'nguyendevwk/sentiment-distillation-kaggle'
PRIVATE_BRANCH = 'main'
GITHUB_AUTH = 'secrets'  # 'secrets': Kaggle/Colab Secrets; 'prompt': ô nhập token ẩn khi chạy.
PROJECT_DIR = BASE / 'private-source' if USE_PRIVATE_REPO else BASE
MODEL = 'zhang-yice/Qwen2.5-1.5B-sentiment-distillation-v1'
# MODEL cũng nhận thư mục model đã attach, ví dụ /kaggle/input/.../model
DATA_DIR = None  # Hoặc đường dẫn chứa sc/, multifaced/, absa/ trong Kaggle Input.
DATASETS = ['sc/sst2']
SEEDS = [42]
MAX_SAMPLES = 32  # 0 = toàn bộ test. Giữ nguyên cấu hình khi tiếp tục checkpoint.
BATCH_SIZE = 1
K_SHOT = 4
MAX_INPUT_TOKENS = 2048
MAX_NEW_TOKENS = 150  # Dùng cho FSA; phân loại sử dụng điểm token đầu.
DEMO_SPLIT = 'train'  # Giữ cách lấy demo của source. 'val' cần val.json ở từng dataset.
FOUR_BIT = False  # Chỉ bật nếu cần giảm VRAM; có cài và kiểm tra bitsandbytes riêng.
DTYPE = 'auto'  # BF16 nếu GPU hỗ trợ, FP16 trên GPU khác.
OUTPUT = BASE / 'output'  # Colab: có thể đổi sang thư mục Google Drive đã mount.
OUTPUT.mkdir(parents=True, exist_ok=True)
EXPERIMENT = 'qwen-notebook'
ENV_DIR = Path('/tmp/sentiment-eval-venv')
PYTHON = ENV_DIR / 'bin/python'
print('Workspace:', BASE, '\nOutput:', OUTPUT)


## GitHub token để clone repo private

**Kaggle:** mở **Add-ons → Secrets**, thêm secret tên `GITHUB_TOKEN`, dán GitHub Personal Access Token
vào **Value** và bật quyền sử dụng secret cho notebook. Tên secret phải đúng là `GITHUB_TOKEN`.
Token fine-grained chỉ cần chọn repo này và cấp **Contents: Read-only** (Metadata đi kèm).
[Tạo token](https://github.com/settings/personal-access-tokens/new).

Hoặc đặt `GITHUB_AUTH='prompt'`: khi chạy sẽ hiện ô nhập token ẩn. Không dán token vào code/config.
Colab có thể dùng Secrets cùng tên (cấp Notebook access), hoặc chọn `prompt`.

Cell clone vào `private-source/`; chạy lại sẽ fetch và cập nhật nhánh bằng fast-forward.
Nếu checkout có file đã sửa, cell dừng để tránh ghi đè. Token chỉ truyền tạm cho tiến trình Git,
không lưu vào URL remote, file credentials, notebook output hoặc log.
Chọn `USE_PRIVATE_REPO=False` để bỏ qua cell này và dùng source đóng gói.


In [ ]:
def clone_private_source():
    import getpass, re, tempfile
    if not re.fullmatch(r'[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+', PRIVATE_REPO):
        raise ValueError('PRIVATE_REPO phải có dạng owner/repo, không chứa token hoặc URL.')
    if not PRIVATE_BRANCH or PRIVATE_BRANCH.startswith('-'):
        raise ValueError('PRIVATE_BRANCH không hợp lệ.')
    if GITHUB_AUTH == 'prompt':
        token = getpass.getpass('GitHub token (ẩn, không lưu vào notebook): ').strip()
    elif GITHUB_AUTH == 'secrets':
        try:
            if Path('/kaggle').exists():
                from kaggle_secrets import UserSecretsClient
                user_secrets = UserSecretsClient()
                secret_value_0 = user_secrets.get_secret("GITHUB_TOKEN")
                token = secret_value_0.strip()
                del secret_value_0
            else:
                from google.colab import userdata
                token = (userdata.get("GITHUB_TOKEN") or '').strip()
        except Exception:
            raise RuntimeError('Không đọc được secret. Thêm GITHUB_TOKEN vào Secrets và cấp quyền cho notebook, '
                               "hoặc đặt GITHUB_AUTH='prompt'.") from None
    else:
        raise ValueError("GITHUB_AUTH chỉ nhận 'secrets' hoặc 'prompt'.")
    if not token:
        raise ValueError('GitHub token trống.')
    url = f'https://github.com/{PRIVATE_REPO}.git'
    git_env = os.environ.copy()
    git_env.update(PRIVATE_CLONE_TOKEN=token, GIT_TERMINAL_PROMPT='0',
                   GIT_CONFIG_NOSYSTEM='1', GIT_CONFIG_GLOBAL=os.devnull)
    # Helper contains no token; the token exists only in the Git subprocess environment.
    with tempfile.TemporaryDirectory(prefix='sentiment-git-auth-') as auth_dir:
        askpass = Path(auth_dir) / 'askpass.sh'
        askpass.write_text(r"""#!/bin/sh
case "$1" in
  *Username*) printf '%s\n' 'x-access-token' ;;
  *) printf '%s\n' "$PRIVATE_CLONE_TOKEN" ;;
esac
""", encoding='utf-8')
        askpass.chmod(0o700)
        git_env['GIT_ASKPASS'] = str(askpass)

        def git(arguments, cwd=None):
            result = subprocess.run(['git', '-c', 'credential.helper=', *arguments], cwd=cwd,
                                    env=git_env, capture_output=True, text=True)
            if result.returncode:
                # No raw stderr/exception: authentication errors must never reveal credentials.
                raise RuntimeError('Git thất bại (không in chi tiết xác thực). Kiểm tra Internet, repo/branch, '
                                   'token còn hạn và quyền Contents: Read-only; checkout phải sạch.') from None
            return result.stdout.strip()

        if (PROJECT_DIR / '.git').is_dir():
            if git(['remote', 'get-url', 'origin'], PROJECT_DIR) != url:
                raise RuntimeError('Checkout hiện tại thuộc repo khác. Chọn workspace khác.')
            if git(['status', '--porcelain'], PROJECT_DIR):
                raise RuntimeError('Checkout có thay đổi cục bộ. Lưu thay đổi hoặc chọn workspace khác trước khi cập nhật.')
            if git(['branch', '--show-current'], PROJECT_DIR) != PRIVATE_BRANCH:
                raise RuntimeError('Checkout đang ở nhánh khác. Đổi PRIVATE_BRANCH hoặc chọn workspace khác.')
            shallow = git(['rev-parse', '--is-shallow-repository'], PROJECT_DIR) == 'true'
            git(['fetch', *(['--unshallow'] if shallow else []), 'origin', PRIVATE_BRANCH], PROJECT_DIR)
            git(['merge', '--ff-only', 'FETCH_HEAD'], PROJECT_DIR)
        else:
            if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
                raise RuntimeError('Thư mục clone đã tồn tại và không rỗng. Chọn workspace khác.')
            git(['clone', '--depth', '1', '--branch', PRIVATE_BRANCH, '--single-branch', url, str(PROJECT_DIR)])
        commit = git(['rev-parse', 'HEAD'], PROJECT_DIR)
    git_env.pop('PRIVATE_CLONE_TOKEN', None)
    token = None
    return commit

if USE_PRIVATE_REPO:
    source_commit = clone_private_source()
    print('Private source ready:', PRIVATE_REPO, 'commit', source_commit)
else:
    print('Using bundled source; GitHub token not needed.')


## Source đi kèm
Cell sau chỉ ghi source đóng gói khi `USE_PRIVATE_REPO=False`; không ghi đè source clone từ repo private.
Các nội dung file được đóng gói trực tiếp bên dưới để notebook hoạt động độc lập.


In [ ]:
PATCHED_SOURCES = {'fsa.py': 'from notebook_runtime import (add_runtime_arguments, prepare_runtime, load_model,\n    select_examples, configure_tokenizer, limit_prompts, run_metadata, PredictionCache)\nfrom tqdm import tqdm\nimport argparse\nfrom utils import save_line_json, load_line_json, save_result\nimport random\nfrom collections import Counter\nfrom torch.utils.data import Dataset, DataLoader\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nimport torch\nfrom utils import load_json,save_json\nfrom sklearn.metrics import f1_score, accuracy_score, classification_report\nimport datetime\nimport time\nimport os\n\nfrom parse_utils.asqp import Result_asqp\nfrom parse_utils.ssa import Result_ssa\nfrom parse_utils.coqe import Result_coqe\nfrom parse_utils.atsa import Result_atsa\nfrom parse_utils.acsa import Result_acsa\n\n\ndef ifnt0(x):\n    """\n    Return 0 if the input is None, otherwise return the input.\n    :param x: The input value that may be None.\n    :return: 0 if x is None, x otherwise.\n    """\n    return 0 if x is None else x\n\nclass FineTuneDataset(Dataset):\n    def __init__(self, dataset_name, dataset, demo_pool ,k_shot, model_name_or_path, mode, max_length):\n        self.model_name_or_path = model_name_or_path\n        self.demo_pool = demo_pool\n        self.k_shot = k_shot\n        self.dataset_name = dataset_name\n        self.dataset_dict = {dataset_name: dataset}\n        self.max_length = max_length\n        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name_or_path, use_fast=True, padding_side=\'left\')\n        self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.mode = mode\n        self.prompts = {\n        \'absa/asqp_rest16\':"""Please perform Aspect Sentiment Quad Prediction task. Given the sentence, extract all (aspect term, aspect category, opinion, sentiment polarity) quadruples.\n\n1. Aspect category should be selected from [\'ambience general\', \'drinks prices\',\'drinks quality\', \'drinks style_options\', \'food general\', \'food prices\', \'food quality\', \'food style_options\', \'location general\', \'restaurant general\', \'restaurant miscellaneous\', \'restaurant pcrices\', \'service general\'].\n\n2. Sentiment polarity should be selected from [\'negative\', \'neutral\', \'positive\'].\n\n3. If there is no aspect term, use \'NULL\' as the aspect term. Only aspect term can be \'NULL\', aspect category, opinion and sentiment polarity CANNOT be \'NULL\'.\n\n4. Please return python list only, without any other comments or texts.""",\n        \'absa/asqp_laptop\':"""Please perform Aspect Sentiment Quad Prediction task. Given the sentence, extract all (aspect term, aspect category, opinion, sentiment polarity) quadruples.\n\n1. Aspect category should be selected from [\'battery miscellaneous\', \'battery operation_performance\', \'battery quality\', \'company general\', \'cpu design_features\', \'cpu miscellaneous\', \'cpu operation_performance\', \'cpu quality\', \'display design_features\', \'display general\', \'display operation_performance\', \'display quality\', \'display usability\', \'fans_cooling design_features\', \'fans_cooling operation_performance\', \'fans_cooling quality\', \'graphics design_features\', \'graphics general\', \'graphics miscellaneous\', \'graphics quality\', \'hard_disc design_features\', \'hard_disc quality\', \'hardware general\', \'hardware operation_performance\', \'hardware quality\', \'keyboard design_features\', \'keyboard general\', \'keyboard operation_performance\', \'keyboard quality\', \'keyboard usability\', \'laptop connectivity\', \'laptop design_features\', \'laptop general\', \'laptop miscellaneous\', \'laptop operation_performance\', \'laptop portability\', \'laptop price\', \'laptop quality\', \'laptop usability\', \'memory design_features\', \'motherboard quality\', \'mouse design_features\', \'mouse general\', \'mouse operation_performance\', \'mouse quality\', \'mouse usability\', \'multimedia_devices design_features\', \'multimedia_devices general\', \'multimedia_devices miscellaneous\', \'multimedia_devices operation_performance\', \'multimedia_devices quality\', \'multimedia_devices usability\', \'optical_drives operation_performance\', \'optical_drives quality\', \'os design_features\', \'os general\', \'os miscellaneous\', \'os operation_performance\', \'os quality\', \'os usability\', \'ports design_features\', \'ports operation_performance\', \'ports quality\', \'power_supply design_features\', \'power_supply miscellaneous\', \'power_supply operation_performance\', \'power_supply quality\', \'shipping price\', \'shipping quality\', \'software design_features\', \'software general\', \'software miscellaneous\', \'software operation_performance\', \'software price\', \'software quality\', \'software usability\', \'support miscellaneous\', \'support price\', \'support quality\', \'warranty general\', \'warranty price\'].\n\n2. Sentiment polarity should be selected from [\'negative\', \'neutral\', \'positive\'].\n\n3. If there is no aspect term, use \'NULL\' as the aspect term. Only aspect term can be \'NULL\', aspect category, opinion and sentiment polarity CANNOT be \'NULL\'.\n\n4. Please return python list only, without any other comments or texts.""",\n\n            \'absa/opener\':"""Please perform the Structured Sentiment Analysis task. Given a sentence, extract all opinion tuples in the format (holder, target, sentiment expression, sentiment polarity). \n\nEach tuple should contain:\n\n- Holder: The entity expressing the sentiment, if there is no explicit holder, use \'NULL\' as the holder.\n\n- Target: The entity being evaluated, if there is no explicit target, use \'NULL\' as the target.\n\n- Sentiment Expression: The phrase conveying the sentiment.\n\n- Sentiment Polarity: The polarity of the sentiment, either positive, negative, or neutral.\n\nFollow these rules:\n\n1. If there is no sentiment expression, return \'NULL\' for all fields.\n\n2. Please return python list only, without any other comments or texts.""",\n\n            \'absa/dsunis\':"""Please perform the Structured Sentiment Analysis task. Given a sentence, extract all opinion tuples in the format (holder, target, sentiment expression, sentiment polarity). \n\nEach tuple should contain:\n\n- Holder: The entity expressing the sentiment, if there is no explicit holder, use \'NULL\' as the holder.\n\n- Target: The entity being evaluated, if there is no explicit target, use \'NULL\' as the target.\n\n- Sentiment Expression: The phrase conveying the sentiment.\n\n- Sentiment Polarity: The polarity of the sentiment, either positive, negative, or neutral.\n\nFollow these rules:\n\n1. If there is no sentiment expression, return \'NULL\' for all fields.\n\n2. Please return python list only, without any other comments or texts.""",\n            \n            \'absa/coqe_camera\':"""Please perform the Comparative Opinion Quintuple Extraction task. Given a sentence, extract all opinion tuples in the format (subject, object, comparative aspect, comparative opinion, comparative preference).\n\nEach tuple should contain:\n\n- subject: The first entity being compared.\n\n- object: The second entity being compared.\n\n- comparative aspect: The feature or attribute being compared.\n\n- comparative opinion: The opinion expression indicating the comparative preference between two entities.\n\n- comparative preference: One of the four possible comparative relationships:\n\n    Worse: The subject is worse than the object.\n\n    Equal: The subject is equal to the object.\n\n    Better: The subject is better than the object.\n\n    Different: The subject is different from the object.\n\nIf\xa0any component is missing, use NULL as its placeholder. Please extract all opinion tuples in the format (subject, object, comparative aspect, comparative opinion, comparative preference). Please return python list only, without any other comments or texts.""",\n    "absa/atsa_rest16":"Please perform Aspect Term Sentiment Analysis task. Given the sentence, extract all (aspect term, sentiment polarity) pairs.",\n    "absa/acsa_rest16":"Please perform aspect-level sentiment analysis task. Given the sentence, tag all (aspect category, sentiment) pairs. Aspect category should be selected from [\'ambience general\', \'drinks prices\', \'drinks quality\', \'drinks style_options\', \'food prices\', \'food quality\', \'food style_options\', \'location general\', \'restaurant general\', \'restaurant miscellaneous\', \'restaurant prices\', \'service general\'], and sentiment should be selected from [\'negative\', \'neutral\', \'positive\']. If there are no target-sentiment pairs, return an empty list. Otherwise return a python list of tuples containing two strings in double quotes. Please return python list only, without any other comments or texts.",\n        }\n        self.construct_sample()\n\n    def __len__(self):\n        return len(self.dataset)\n\n\n    def add_prompt(self, dataset_name, data):\n\n        prompt = self.prompts[dataset_name]\n        prompts = [prompt]\n        prompts.append(\'\')\n\n\n        sentence = data[\'sentence\']\n        demos = random.sample(self.demo_pool, k=self.k_shot)\n\n        for demo in demos:\n            prompts.append(f"Sentence: {demo[\'sentence\']}\\nLabel: {demo[\'label_seq\']}\\n")\n\n        \n        prompts.append(f"Sentence: {sentence}\\nLabel:")\n        data[\'prompts\'] = \'\\n\'.join(prompts)\n        data[\'dataset\'] = dataset_name\n\n\n    def make_label_seq(self, dataset_name, data):\n\n        def make_atsa_rest16_seq(example):\n            if \'label_seq\' in example:\n                return example[\'label_seq\']\n\n            atsa_seq = []\n            for aspect in example[\'aspects\']:\n                polarity = aspect[\'polarity\']\n                aspect_term = aspect[\'target\']\n\n                if aspect_term in (\'NULL\', None):\n                    continue\n\n                assert aspect_term in example[\'sentence\']\n\n                loc = (ifnt0(aspect[\'from\']), ifnt0(aspect[\'to\']))\n\n                atsa_seq.append((aspect_term, polarity, loc))\n\n            atsa_seq = list(set(atsa_seq))\n            atsa_seq = sorted(atsa_seq, key=lambda it: it[-1])\n            atsa_seq = [(aspect_term, polarity) for aspect_term, polarity, _ in atsa_seq]\n            example[\'label_seq\'] = atsa_seq\n\n\n        def make_acsa_rest16_seq(example):\n            if \'label_seq\' in example:\n                return example[\'label_seq\']\n\n\n            acsa_seq = []\n            for aspect in example[\'aspects\']:\n                polarity = aspect[\'polarity\']\n                category = aspect[\'category\'].lower().replace(\'#\', \' \')\n\n                category_polarity = (category, polarity)\n                if category_polarity not in acsa_seq:\n                    acsa_seq.append(category_polarity)\n\n            example[\'label_seq\'] = acsa_seq\n\n\n        def make_asqp_seq(example):\n            if \'label_seq\' in example:\n                print(1)\n                return example[\'label_seq\']\n\n\n            asqp_seq = []\n            for quard in example[\'quards\']:\n\n                aspect = quard[\'aspect\']\n                category = quard[\'category\']\n                sentiment = quard[\'sentiment\']\n                opinion = quard[\'opinion\']\n\n\n\n                asqp_quard = (category, aspect, opinion, sentiment)\n                if asqp_quard not in asqp_seq:\n                    asqp_seq.append(asqp_quard)\n\n            example[\'label_seq\'] = asqp_seq\n\n\n\n        def make_ssa_seq(example):\n            if \'label_seq\' in example:\n                return example[\'label_seq\']\n\n            ssa_seq = []\n            for opinion in example[\'opinions\']:\n\n                if len(opinion[\'Source\'][0]) == 0:\n                    holder = \'NULL\'\n                else:\n                    holder = \' AND \'.join(opinion[\'Source\'][0])\n\n                if len(opinion[\'Target\'][0]) == 0:\n                    target = \'NULL\'\n                else:\n                    target = \' AND \'.join(opinion[\'Target\'][0])\n\n                expressions = opinion[\'Polar_expression\'][0]\n                expressions = \' AND \'.join(expressions)\n\n                sentiment = opinion[\'Polarity\'].lower()\n\n                sst_quard = (holder, target, expressions, sentiment)\n                if sst_quard not in ssa_seq:\n                    ssa_seq.append(sst_quard)\n\n            if example[\'opinions\'] == []:\n                ssa_seq.append((\'NULL\', \'NULL\', \'NULL\', \'NULL\'))\n\n            example[\'label_seq\'] = ssa_seq\n\n        def make_multifaced_seq(example):\n            if \'label_seq\' in example:\n                return example[\'label_seq\']\n            example[\'label_seq\'] = example[\'label\'].lower()\n\n        def make_coqe_seq(example):\n            if \'label_seq\' in example:\n                return example[\'label_seq\']\n\n\n            coqe_seq = []\n            for quintuple in example[\'quintuple\']:\n\n                subject = quintuple[\'subject\'] if quintuple[\'subject\'] != \'\' else \'NULL\'\n                object = quintuple[\'object\'] if quintuple[\'object\'] != \'\' else \'NULL\'\n                aspect = quintuple[\'comparative aspect\'] if quintuple[\'comparative aspect\'] != \'\' else \'NULL\'\n                opinion = quintuple[\'comparative opinion\'] if quintuple[\'comparative opinion\'] != \'\' else \'NULL\'\n                preference = quintuple[\'comparative preference\'] if quintuple[\'comparative preference\'] != \'\' else \'NULL\'\n\n\n                coqe_quard = (subject, object, aspect, opinion, preference)\n                if coqe_quard not in coqe_seq:\n                    coqe_seq.append(coqe_quard)\n\n            example[\'label_seq\'] = coqe_seq\n\n\n\n\n        if dataset_name == \'absa/asqp_rest16\' or dataset_name == \'absa/asqp_laptop\':\n            make_asqp_seq(data)\n        elif dataset_name == \'absa/dsunis\' or dataset_name == \'absa/opener\':\n            make_ssa_seq(data)\n        elif dataset_name == \'absa/coqe_camera\':\n            make_coqe_seq(data)\n        elif dataset_name == \'absa/atsa_rest16\':\n            make_atsa_rest16_seq(data)\n        elif dataset_name == \'absa/acsa_rest16\':\n            make_acsa_rest16_seq(data)\n        else:\n            make_multifaced_seq(data)\n\n\n    def construct_sample(self):\n\n\n        print(self.dataset_name)\n        for item in self.demo_pool:\n            self.make_label_seq(self.dataset_name, item)\n\n\n        for dataset_name, dataset in self.dataset_dict.items():\n            for data in dataset:\n                self.add_prompt(dataset_name, data)\n                self.make_label_seq(dataset_name, data)\n\n\n        self.dataset = []\n        for dataset in self.dataset_dict.values():\n            self.dataset.extend(dataset)\n        self.dataset = self.dataset\n\n\n\n    def __getitem__(self, idx):\n        item = self.dataset[idx]\n        prompt = item[\'prompts\']\n        label_seq = item[\'label_seq\']\n\n        return item\n\n\n\ndef cal_metric(results, f1, seed, subname, model_name_or_path, dataset, output_dir, k_shot, metadata=None):\n    model_name = model_name_or_path.split(\'/\')[-1]\n    print(model_name)\n    current_time = time.strftime(\'%Y_%m_%d_%H_%M_%S\')\n    file_name = f\'{subname}_{model_name}_{dataset}_{seed}_numOfshot{k_shot}_{current_time}\'\n    output_path = os.path.join(output_dir, file_name + \'.json\')\n    print(output_path)\n\n\n\n\n    save_result(output_dir, file_name, f1, 0, metadata)\n    save_json(results, output_path)\n\n\nclass DataCollator:\n    def __init__(self, tokenizer, max_seq_length, mode):\n        self.tokenizer = tokenizer\n        self.max_seq_length = max_seq_length\n        self.mode = mode\n\n    def tok(self, text, max_seq_length):\n        kwargs = {\n            \'text\': text,\n            \'return_tensors\': \'pt\',\n        }\n\n        if max_seq_length in (-1, \'longest\'):\n            kwargs[\'padding\'] = True\n\n        else:\n            kwargs[\'max_length\'] = max_seq_length\n            kwargs[\'padding\'] = True\n            kwargs[\'truncation\'] = True\n\n        batch_encodings = self.tokenizer(**kwargs)\n        return batch_encodings\n\n    def __call__(self, examples):\n\n        texts = [example[\'prompts\'] for example in examples]\n        batch_encodings = self.tok(texts, self.max_seq_length)\n\n        input_ids = batch_encodings[\'input_ids\']\n        attention_mask = batch_encodings[\'attention_mask\']\n\n        return {\n            \'input_ids\': input_ids,\n            \'attention_mask\': attention_mask,\n            \'examples\': examples\n        }\n\ndef get_dataloader(test_examples, mode, batch_size, shuffle, tokenizer):\n    dataloader = DataLoader(\n        dataset=test_examples,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        pin_memory=True,\n        num_workers=0,\n        collate_fn=DataCollator(\n            tokenizer=tokenizer,\n            max_seq_length=max_seq_length,\n            mode=mode,\n        )\n    )\n\n    print(\'dataloader-\' + mode, len(dataloader))\n    return dataloader\n\n\ndef test_dataloader(test_examples, eval_batch_size,tokenizer):\n    return get_dataloader(test_examples, "test", eval_batch_size, shuffle=False, tokenizer=tokenizer)\n\n\n# 从test.json 和 train.json 中加载数据\ndef load_dataset(data_path, demo_split=\'train\'):\n    test_file_name = os.path.join(data_path, \'test.json\')\n    test_examples = load_json(test_file_name)\n\n    demo_pool_file_name = os.path.join(data_path, demo_split + \'.json\')\n    demo_examples = load_json(demo_pool_file_name)\n\n    raw_datasets = {\n        \'test\': test_examples,\n        \'demo\': demo_examples\n    }\n\n    print(\'-----------data statistic-------------\')\n    for mode in (\'test\',\'demo\'):\n        num_samples = len(raw_datasets[mode])\n        print(f\'{mode.upper():<5} | Samples: {num_samples:<5}\')\n    print(\'--------------------------------------\')\n\n\n\n    return raw_datasets\n\n\n\nimport sys\nimport argparse\nimport datetime\nif __name__ == \'__main__\':\n    print(\'hello\')\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument(\'--dataset\', required=True, )\n    parser.add_argument(\'--subname\', required=True, )\n    parser.add_argument(\'--seed\', required=True, type=int)\n    parser.add_argument(\'--model_name_or_path\', default=None )\n    parser.add_argument(\'--model_version\', default=None)\n    parser.add_argument(\'--max_seq_length\', type=int, default=2048)\n    parser.add_argument(\'--data_prop\', type=float, default=1)\n    parser.add_argument(\'--eval_batch_size\', type=int, default=1)\n    parser.add_argument(\'--data_dir\', default=os.path.join(os.path.dirname(__file__), \'datasets\'))\n    parser.add_argument(\'--output_dir\', default=\'./output\')\n    parser.add_argument(\'--k_shot\', type=int, default=4)\n\n    add_runtime_arguments(parser)\n    args = parser.parse_args()\n    prepare_runtime(args)\n\n    dataset = args.dataset\n    subname = args.subname\n    seed = args.seed\n    model_version = args.model_version\n    model_name_or_path = args.model_name_or_path\n    max_seq_length = args.max_seq_length\n    data_prop = args.data_prop\n    eval_batch_size = args.eval_batch_size\n    data_dir = args.data_dir\n    output_dir = args.output_dir\n    k_shot = args.k_shot\n\n    data_path = os.path.join(data_dir, dataset)\n    torch.manual_seed(seed)\n    random.seed(seed)\n\n    torch.cuda.manual_seed(seed)\n\n\n    model_name_or_path = args.model_name_or_path\n    print(\'model name: \', model_name_or_path)\n    print(\'-\'*20+\'Experiment Setup\'+\'-\'*20)\n    print(\'data_path: \', data_path)\n    print(f\' subname: {subname}\\n dataset: {dataset}\\n seed:{seed}\\n model_name_or_path:{model_name_or_path}\\n data_path:{data_path}\\n output_dir:{output_dir}\\n eval_batch_size:{eval_batch_size}\\n data_prop:{data_prop}\\n max_seq_length:{max_seq_length}\\n numbers_of_shot:{k_shot}\')\n    print(\'-\' * 50)\n\n\n    raw_datasets = select_examples(load_dataset(data_path, args.demo_split), args)\n\n\n\n    test_examples = FineTuneDataset(dataset, raw_datasets[\'test\'], raw_datasets[\'demo\'], k_shot, model_name_or_path, \'test\', max_seq_length)\n    \n\n    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, padding_side=\'left\')\n    tokenizer.pad_token = tokenizer.eos_token\n\n\n    \n    model = load_model(args)\n    configure_tokenizer(tokenizer, model, args, args.max_new_tokens)\n    max_seq_length = args.max_seq_length\n    truncated = limit_prompts(test_examples, tokenizer, max_seq_length)\n    metadata = run_metadata(args, raw_datasets, truncated)\n    cache = PredictionCache(args.resume_file, metadata)\n    pending = [example for example in test_examples if cache.get(example) is None]\n    dataloader = test_dataloader(pending, eval_batch_size, tokenizer)\n\n\n    print(model_name_or_path)\n    model.eval()\n\n    with torch.inference_mode():\n        for batch in tqdm(dataloader):\n            input_ids = batch[\'input_ids\'].to(model.device)\n            generated_ids = model.generate(\n                input_ids,\n                attention_mask=batch[\'attention_mask\'].to(model.device),\n                max_new_tokens=args.max_new_tokens,\n                num_beams=1, num_return_sequences=1, do_sample=False,\n                pad_token_id=tokenizer.pad_token_id,\n            )\n            # Decode only new tokens, independent of prompt truncation and shot count.\n            generateds = tokenizer.batch_decode(\n                generated_ids[:, input_ids.shape[1]:], skip_special_tokens=True)\n            for example, prediction in zip(batch[\'examples\'], generateds):\n                cache.put(example, prediction.strip())\n            cache.flush()\n            del generated_ids, input_ids\n\n    outputs = [(example, cache.get(example)) for example in test_examples]\n\n    current_results = []\n\n    if  dataset == \'absa/asqp_rest16\' or dataset == \'absa/asqp_laptop\': \n        current_results.append(Result_asqp.parse_from(outputs))\n    elif dataset == \'absa/opener\' or dataset == \'absa/dsunis\':\n        current_results.append(Result_ssa.parse_from(outputs))\n    elif dataset == \'absa/coqe_camera\':\n        current_results.append(Result_coqe.parse_from(outputs))\n    elif dataset == \'absa/atsa_rest16\':\n        current_results.append(Result_atsa.parse_from(outputs))\n    elif dataset == \'absa/acsa_rest16\':\n        current_results.append(Result_acsa.parse_from(outputs))\n    else:\n        raise ValueError(f\'No dataset named {dataset}\')\n\n\n    for result in current_results:\n        result.cal_metric()\n\n\n\n\n    current_results[0].report()\n    f1_scores = current_results[0].monitor\n    print(f1_scores)\n    \n    \n    \n    cal_metric(outputs,f1_scores, seed, subname, model_name_or_path, dataset, output_dir, k_shot, metadata)\n', 'icl_multifaced.py': 'from notebook_runtime import (add_runtime_arguments, prepare_runtime, load_model,\n    select_examples, configure_tokenizer, limit_prompts, run_metadata, PredictionCache)\nfrom tqdm import tqdm\nimport argparse\nfrom utils import save_line_json, load_line_json, save_result\nimport random\nfrom collections import Counter\nfrom torch.utils.data import Dataset, DataLoader\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nimport torch\nfrom utils import load_json,save_json\nfrom sklearn.metrics import f1_score, accuracy_score, classification_report\nimport datetime\nimport time\nimport os\n\n\nid_to_label = {\n    \'multifaced/tweeteval\': [\' anger\', \' joy\', \' sadness\', \' optimism\'],\n    \'multifaced/intimacy\': [\' not intimate\', \' moderately intimate\', \' highly intimate\'],\n    \'multifaced/offenseval\':[\' non-offensive\', \' offensive\'],\n    \'multifaced/irony18\': [\' non-irony\', \' irony\'],\n    \'multifaced/pstance\': [\' against\', \' favor\'],\n    \'sc/imdb\': [\' negative\', \' positive\'],\n    \'sc/sst2\': [\' negative\', \' positive\'],\n    \'sc/yelp2\': [\' negative\', \' positive\'],\n    \'sc/twitter\': [\' negative\', \' positive\', \' neutral\'],\n}\n\n\ndef get_label_list(tokenizer, dataset):\n    ids = [tokenizer(label, add_special_tokens=False)[\'input_ids\'][0]\n           for label in id_to_label[dataset]]\n    if len(set(ids)) != len(ids):\n        raise ValueError(\'Label first tokens collide for this tokenizer; use a compatible model or implement full-label scoring.\')\n    return ids\n\n\ndef cal_metric(results, y_true, y_pred, seed, subname, model_name_or_path, dataset, output_dir, k_shot, metadata=None):\n    model_name = model_name_or_path.split(\'/\')[-1]\n    print(model_name)\n    current_time = time.strftime(\'%Y_%m_%d_%H_%M_%S\')\n    file_name = f\'{subname}_{model_name}_{dataset}_{seed}_numOfshot{k_shot}_{current_time}\'\n    output_path = os.path.join(output_dir, file_name + \'.json\')\n    print(output_path)\n\n    print(Counter(y_true))\n    print(Counter(y_pred))\n\n    f1 = f1_score(y_true, y_pred, average=\'macro\')\n    accuracy = accuracy_score(y_true, y_pred)\n    print(f\'f1: {f1:.4f} | accuracy: {accuracy:.4f}\\n\')\n\n\n    save_result(output_dir, file_name, f1, accuracy, metadata)\n    save_json(results, output_path)\n\n\nclass DataCollator:\n    def __init__(self, tokenizer, max_seq_length, mode):\n        self.tokenizer = tokenizer\n        self.max_seq_length = max_seq_length\n        self.mode = mode\n\n    def tok(self, text, max_seq_length):\n        kwargs = {\n            \'text\': text,\n            \'return_tensors\': \'pt\',\n        }\n\n        if max_seq_length in (-1, \'longest\'):\n            kwargs[\'padding\'] = True\n\n        else:\n            kwargs[\'max_length\'] = max_seq_length\n            kwargs[\'padding\'] = True\n            kwargs[\'truncation\'] = True\n\n        batch_encodings = self.tokenizer(**kwargs)\n        return batch_encodings\n\n    def __call__(self, examples):\n\n        texts = [example[\'prompts\'] for example in examples]\n        batch_encodings = self.tok(texts, self.max_seq_length)\n\n        input_ids = batch_encodings[\'input_ids\']\n        attention_mask = batch_encodings[\'attention_mask\']\n\n        return {\n            \'input_ids\': input_ids,\n            \'attention_mask\': attention_mask,\n            \'examples\': examples\n        }\n\ndef get_dataloader(test_examples, mode, batch_size, shuffle, tokenizer):\n    dataloader = DataLoader(\n        dataset=test_examples,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        pin_memory=True,\n        num_workers=0,\n        collate_fn=DataCollator(\n            tokenizer=tokenizer,\n            max_seq_length=max_seq_length,\n            mode=mode,\n        )\n    )\n\n    print(\'dataloader-\' + mode, len(dataloader))\n    return dataloader\n\n\ndef test_dataloader(test_examples, eval_batch_size,tokenizer):\n    return get_dataloader(test_examples, "test", eval_batch_size, shuffle=False, tokenizer=tokenizer)\n\n\ndef construct_prompt(prompt, test_examples, demo_pool, k_shot):\n\n    for example in test_examples:\n        text = example[\'sentence\']\n        demos = random.sample(demo_pool, k=k_shot)\n        prompts = [prompt]\n        prompts.append(\'\')\n        for demo in demos:\n            prompts.append(f"Sentence: {demo[\'sentence\']}\\nLabel: {demo[\'label\'].lower()}\\n")\n        prompts.append(f"Sentence: {example[\'sentence\']}\\nLabel:")\n        example[\'prompts\'] = \'\\n\'.join(prompts)\n\n\n\n    return test_examples\n\n\ndef construct_prompt_pstance(prompt, test_examples, demo_pool, k_shot):\n\n    for example in test_examples:\n        text = example[\'sentence\']\n        target = example[\'target\']\n        demos = random.sample(demo_pool, k=k_shot)\n        prompts = [prompt % target]\n        prompts.append(\'\')\n        for demo in demos:\n            prompts.append(f"Sentence: {demo[\'sentence\']}\\nLabel: {demo[\'label\'].lower()}(opinion towards \'{demo[\'target\']}\')\\n")\n        prompts.append(f"Sentence: {example[\'sentence\']}\\nLabel:")\n        example[\'prompts\'] = \'\\n\'.join(prompts)\n\n\n\n    return test_examples\n\n\n\ndef load_dataset(data_path, demo_split=\'train\'):\n    test_file_name = os.path.join(data_path, \'test.json\')\n    test_examples = load_json(test_file_name)\n\n    demo_pool_file_name = os.path.join(data_path, demo_split + \'.json\')\n    demo_examples = load_json(demo_pool_file_name)\n\n    raw_datasets = {\n        \'test\': test_examples,\n        \'demo\': demo_examples\n    }\n\n    print(\'-----------data statistic-------------\')\n    for mode in (\'test\',\'demo\'):\n        num_samples = len(raw_datasets[mode])\n        print(f\'{mode.upper():<5} | Samples: {num_samples:<5}\')\n    print(\'--------------------------------------\')\n\n\n\n    return raw_datasets\n\n\n\nimport sys\nimport argparse\nimport datetime\nif __name__ == \'__main__\':\n    print(\'hello\')\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument(\'--dataset\', required=True, )\n    parser.add_argument(\'--subname\', required=True, )\n    parser.add_argument(\'--seed\', required=True, type=int)\n    parser.add_argument(\'--model_name_or_path\', default=None )\n    parser.add_argument(\'--model_version\', default=None)\n    parser.add_argument(\'--max_seq_length\', type=int, default=2048)\n    parser.add_argument(\'--data_prop\', type=float, default=1)\n    parser.add_argument(\'--eval_batch_size\', type=int, default=1)\n    parser.add_argument(\'--data_dir\', default=os.path.join(os.path.dirname(__file__), \'datasets\'))\n    parser.add_argument(\'--output_dir\', default=\'./output\')\n    parser.add_argument(\'--k_shot\', type=int, default=4)\n\n    add_runtime_arguments(parser)\n    args = parser.parse_args()\n    prepare_runtime(args)\n\n    dataset = args.dataset\n    subname = args.subname\n    seed = args.seed\n    model_version = args.model_version\n    model_name_or_path = args.model_name_or_path\n    max_seq_length = args.max_seq_length\n    data_prop = args.data_prop\n    eval_batch_size = args.eval_batch_size\n    data_dir = args.data_dir\n    output_dir = args.output_dir\n    k_shot = args.k_shot\n\n    data_path = os.path.join(data_dir, dataset)\n    torch.manual_seed(seed)\n    random.seed(seed)\n\n    torch.cuda.manual_seed(seed)\n\n\n    model_name_or_path = args.model_name_or_path\n\n    print(\'-\'*20+\'Experiment Setup\'+\'-\'*20)\n    print(\'data_path: \', data_path)\n    print(f\' subname: {subname}\\n dataset: {dataset}\\n seed:{seed}\\n model_name_or_path:{model_name_or_path}\\n data_path:{data_path}\\n output_dir:{output_dir}\\n eval_batch_size:{eval_batch_size}\\n data_prop:{data_prop}\\n max_seq_length:{max_seq_length}\\n numbers_of_shot:{k_shot}\')\n    print(\'-\' * 50)\n\n    prompts = {\n        \'multifaced/irony18\':\n            """Please perform Irony Detection task. Given the sentence, assign a sentiment label from [\'irony\', \'non-irony\']. Return label only without any other text.""",\n        \'multifaced/offenseval\':\n            """Please perform Offensive Detection task. Given the sentence, assign a sentiment label from [\'non-offensive\', \'offensive\']. Return label only without any other text.""",\n        \'multifaced/intimacy\':\n            """Please perform Intimacy Detection task. Given the sentence, assign an intimacy label from [\'not intimate\', \'moderately intimate\', \'highly intimate\']. Return label only without any other text.""",\n        \'multifaced/tweeteval\':\n            """Please perform Emotion Detection task. Given the sentence, assign a emotion label from [\'anger\', \'joy\', \'sadness\', \'optimism\']. Return the label only without any other text.""",\n        \'multifaced/pstance\':\n            """Please perform Stance Detection task. Given the sentence, assign a sentiment label expressed by the author towards "%s" from [\'against\', \'favor\']. Return label only without any other text.""",\n        \'sc/imdb\':\n            """Please perform Sentiment Analysis task. Given the sentence, assign a sentiment polarity label from [\'negative\', \'positive\']. Return label only without any other text.""",\n        \'sc/sst2\':\n            """Please perform Sentiment Analysis task. Given the sentence, assign a sentiment polarity label from [\'negative\', \'positive\']. Return label only without any other text.""",\n        \'sc/yelp2\':\n            """Please perform Sentiment Analysis task. Given the sentence, assign a sentiment polarity label from [\'negative\', \'positive\']. Return label only without any other text.""",\n        \'sc/twitter\':\n            """Please perform Sentiment Analysis task. Given the sentence, assign a sentiment polarity label from [\'negative\', \'positive\', \'neutral\']. Return label only without any other text.""",\n    }\n\n\n\n\n    raw_datasets = select_examples(load_dataset(data_path, args.demo_split), args)\n\n    if dataset == \'multifaced/pstance\':\n        test_examples = construct_prompt_pstance(prompts[dataset], raw_datasets[\'test\'], raw_datasets[\'demo\'], k_shot)\n    else:\n        test_examples = construct_prompt(prompts[dataset], raw_datasets[\'test\'], raw_datasets[\'demo\'], k_shot)\n\n    for i in range(1):\n        print(test_examples[i][\'prompts\'])\n        print(\'-\'*100)\n\n    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, padding_side=\'left\')\n    tokenizer.pad_token = tokenizer.eos_token\n\n\n\n    model = load_model(args)\n    configure_tokenizer(tokenizer, model, args, 1)\n    max_seq_length = args.max_seq_length\n    truncated = limit_prompts(test_examples, tokenizer, max_seq_length)\n    metadata = run_metadata(args, raw_datasets, truncated)\n    cache = PredictionCache(args.resume_file, metadata)\n    pending = [example for example in test_examples if cache.get(example) is None]\n    dataloader = test_dataloader(pending, eval_batch_size, tokenizer)\n\n\n\n\n\n    model.eval()\n\n    label_ids = get_label_list(tokenizer, dataset)\n    results = []\n    with torch.inference_mode():\n        for batch in tqdm(dataloader):\n            # Same first-token decision rule as the original evaluator; no KV cache.\n            output = model(\n                input_ids=batch[\'input_ids\'].to(model.device),\n                attention_mask=batch[\'attention_mask\'].to(model.device),\n                use_cache=False,\n            )\n            scores = output.logits[:, -1, label_ids].float().cpu()\n            for example, score in zip(batch[\'examples\'], scores):\n                cache.put(example, {\'prediction\': id_to_label[dataset][score.argmax().item()].strip(),\n                                    \'prediction_score\': score.tolist()})\n            cache.flush()\n            del output, scores\n\n    for example in test_examples:\n        result = dict(example)\n        result.update(cache.get(example))\n        results.append(result)\n    labels = [example[\'label\'].lower() for example in results]\n    predictions = [example[\'prediction\'] for example in results]\n    cal_metric(results, labels, predictions, seed, subname, model_name_or_path,\n               dataset, output_dir, k_shot, metadata)\n', 'utils.py': 'import os\nimport numpy as np\nimport json\nfrom tqdm import trange\nimport random\n\n\ndef save_result(output_dir, exper_name, f1, acc, metadata=None):\n    result = (exper_name,{\'f1\':round(f1, 4), \'acc\':round(acc, 4)})\n    if metadata is not None:\n        result[1][\'metadata\'] = metadata\n    os.makedirs(output_dir, exist_ok=True)\n    with open(os.path.join(output_dir,\'result.txt\'), \'a\', encoding=\'utf-8-sig\') as f:\n        data = json.dumps(result)\n        f.write(data)\n        f.write(\'\\n\')\n\n\n\ndef tgenerate_batch(examples, bz):\n    # 100//10+0 = 10\n    # 101//10+1 = 11\n    n_batch = len(examples) // bz + int(len(examples) % bz > 0)\n    for i in trange(n_batch):\n        yield examples[i*bz: (i+1)*bz]\n\n\n\ndef params_count(model):\n    """\n    Compute the number of parameters.\n    Args:\n        model (model): model to count the number of parameters.\n    """\n    return np.sum([p.numel() for p in model.parameters()]).item()\n\n\ndef load_text(file_name):\n    with open(file_name, mode=\'r\', encoding=\'utf-8-sig\') as f:\n        for line in f:\n            yield line.strip()\n\n\ndef load_json(file_name):\n    with open(file_name, mode=\'r\', encoding=\'utf-8-sig\') as f:\n        return json.load(f)\n\n\n\ndef load_line_json(file_name):\n    with open(file_name, mode=\'r\', encoding=\'utf-8-sig\') as f:\n        for line in f:\n            yield json.loads(line)\n\n\ndef save_line_json(json_obj, file_name):\n    mkdir_if_not_exist(file_name)\n    with open(file_name, mode=\'w\', encoding=\'utf-8-sig\') as f:\n        lines = [json.dumps(line)+\'\\n\' for line in json_obj]\n        lines[-1] = lines[-1][:-1]\n        f.writelines(lines)\n\n\ndef mkdir_if_not_exist(path):\n    dir_name, file_name = os.path.split(path)\n    if dir_name:\n        if not os.path.exists(dir_name):\n            os.makedirs(dir_name)\n\n\n\nclass NpEncoder(json.JSONEncoder):\n    def default(self, obj):\n        if isinstance(obj, np.integer):\n            return int(obj)\n        elif isinstance(obj, np.floating):\n            return float(obj)\n        elif isinstance(obj, np.ndarray):\n            return obj.tolist()\n        else:\n            return super(NpEncoder, self).default(obj)\n\n\ndef save_json(json_obj, file_name):\n    mkdir_if_not_exist(file_name)\n    with open(file_name, mode=\'w\', encoding=\'utf-8-sig\') as f:\n        json.dump(json_obj, f, indent=4, cls=NpEncoder)\n\n\n\ndef append_json(file_name, obj, mode=\'a\'):\n    mkdir_if_not_exist(file_name)\n    with open(file_name, mode=mode, encoding=\'utf-8\') as f:\n        if type(obj) is dict:\n            string = json.dumps(obj)\n        elif type(obj) is list:\n            string = \' \'.join([str(item) for item in obj])\n        elif type(obj) is str:\n            string = obj\n        else:\n            raise Exception()\n\n        string = string + \'\\n\'\n        f.write(string)\n\n\ndef append_new_line(file_name, string):\n    mkdir_if_not_exist(file_name)\n    with open(file_name, mode=\'a\', encoding=\'utf-8\') as f:\n        string = string + \'\\n\'\n        f.write(string)\n\n\n\ndef yield_data_file(data_dir):\n    for file_name in os.listdir(data_dir):\n        yield os.path.join(data_dir, file_name)\n\n\n\ndef random_round(num):\n    n = int(num)\n    n = n + int(random.random()<(num-n))\n    return n\n', 'parse.py': '"""Summarize modern result metadata and legacy SentiBench result names."""\nimport argparse\nfrom collections import defaultdict\nimport json\nfrom pathlib import Path\nimport re\n\nDATASETS = [\'sc/imdb\', \'sc/yelp2\', \'sc/sst2\', \'sc/twitter\',\n            \'multifaced/irony18\', \'multifaced/tweeteval\', \'multifaced/pstance\',\n            \'multifaced/intimacy\', \'absa/asqp_rest16\', \'absa/opener\',\n            \'absa/atsa_rest16\', \'absa/acsa_rest16\']\n\n\ndef summarize(path, experiments=None):\n    groups = defaultdict(lambda: defaultdict(dict))\n    for line in Path(path).read_text(encoding=\'utf-8-sig\').splitlines():\n        if not line.strip():\n            continue\n        name, metrics = json.loads(line)\n        metadata = metrics.get(\'metadata\')\n        if metadata:\n            experiment = metadata[\'experiment\']\n            dataset = metadata[\'dataset\']\n            settings = json.dumps(metadata[\'settings\'], sort_keys=True)\n            seed = metadata[\'seed\']\n        else:\n            match = re.fullmatch(r\'(.+)_((?:sc|multifaced|absa)/.+)_(-?\\d+)_numOfshot(\\d+)_(\\d{4}(?:_\\d{2}){5})\', name)\n            if not match:\n                raise ValueError(f\'Unrecognized legacy result name: {name}\')\n            experiment, dataset, seed, shots, _ = match.groups()\n            settings = json.dumps({\'legacy\': True, \'k_shot\': int(shots)}, sort_keys=True)\n        if experiments and not any(e in experiment for e in experiments):\n            continue\n        # Re-running the same seed replaces its older result.\n        groups[(experiment, settings)][dataset][str(seed)] = metrics[\'f1\']\n    if not groups:\n        print(\'No completed evaluation results found.\')\n    for (experiment, settings), datasets in groups.items():\n        print(f\'\\nExperiment: {experiment}\\nSettings: {settings}\')\n        means = []\n        for dataset in DATASETS:\n            values = list(datasets.get(dataset, {}).values())\n            if values:\n                mean = sum(values) / len(values) * 100\n                means.append(mean)\n                print(f\'{dataset:<26} {mean:7.2f}  ({len(values)} seed(s))\')\n            else:\n                print(f\'{dataset:<26}     N/A\')\n        if len(means) == len(DATASETS):\n            print(f\'SentiBench average: {sum(means) / len(means):.2f}\')\n        elif means:\n            print(f\'Partial average ({len(means)}/12 datasets): {sum(means) / len(means):.2f}; not the full benchmark score\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--path\', default=\'./output/result.txt\')\n    parser.add_argument(\'--experiments\', nargs=\'*\')\n    args = parser.parse_args()\n    summarize(args.path, args.experiments)\n', 'notebook_runtime.py': '"""Shared, memory-conscious evaluation support for hosted notebooks."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport warnings\n\n\ndef add_runtime_arguments(parser):\n    parser.add_argument(\'--dtype\', choices=[\'auto\', \'float16\', \'bfloat16\', \'float32\'], default=\'auto\')\n    parser.add_argument(\'--load_in_4bit\', action=\'store_true\')\n    parser.add_argument(\'--max_samples\', type=int, default=0, help=\'0: all; otherwise first N test examples\')\n    parser.add_argument(\'--demo_split\', choices=[\'train\', \'val\'], default=\'train\')\n    parser.add_argument(\'--max_new_tokens\', type=int, default=150)\n    parser.add_argument(\'--resume_file\', help=\'Atomic prediction checkpoint; use a persistent output path\')\n\n\ndef prepare_runtime(args):\n    import torch\n    if args.eval_batch_size < 1 or args.k_shot < 0 or args.max_samples < 0:\n        raise ValueError(\'batch_size must be positive; k_shot and max_samples must be nonnegative\')\n    if args.max_seq_length != -1 and args.max_seq_length < 1:\n        raise ValueError(\'max_seq_length must be -1 or a positive token count\')\n    if not 0 < args.data_prop <= 1 or args.max_new_tokens < 1:\n        raise ValueError(\'data_prop must be in (0, 1]; max_new_tokens must be positive\')\n    if args.model_name_or_path:\n        model_path = args.model_name_or_path\n    else:\n        mapping = json.loads(Path(__file__).with_name(\'model_name.json\').read_text())\n        model_path = mapping.get(args.model_version, args.model_version)\n        if not model_path:\n            raise ValueError(\'Provide --model_name_or_path or --model_version\')\n        if model_path.startswith(\'.\'):\n            model_path = str(Path(__file__).parent / model_path)\n    args.model_name_or_path = model_path\n    if args.load_in_4bit and not torch.cuda.is_available():\n        raise ValueError(\'4-bit evaluation requires an available CUDA GPU\')\n    dtype = args.dtype\n    if dtype == \'auto\':\n        dtype = (\'bfloat16\' if torch.cuda.is_bf16_supported() else \'float16\') if torch.cuda.is_available() else \'float32\'\n    if dtype == \'bfloat16\' and (not torch.cuda.is_available() or not torch.cuda.is_bf16_supported()):\n        raise ValueError(\'This GPU does not support BF16; use --dtype auto or float16\')\n    if not torch.cuda.is_available() and dtype == \'float16\':\n        raise ValueError(\'Use float32 for CPU evaluation\')\n    args.resolved_dtype = dtype\n    Path(args.output_dir).mkdir(parents=True, exist_ok=True)\n    print(f\'Runtime: {"CUDA" if torch.cuda.is_available() else "CPU"}, dtype={dtype}, 4bit={args.load_in_4bit}\')\n    return model_path\n\n\ndef load_model(args):\n    import torch\n    from transformers import AutoModelForCausalLM, BitsAndBytesConfig\n    options = dict(torch_dtype=getattr(torch, args.resolved_dtype), low_cpu_mem_usage=True)\n    if torch.cuda.is_available():\n        # One GPU avoids accidental CPU offload or combining incompatible notebook GPUs.\n        options[\'device_map\'] = {\'\': 0}\n    if args.load_in_4bit:\n        options[\'quantization_config\'] = BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=getattr(torch, args.resolved_dtype))\n    model = AutoModelForCausalLM.from_pretrained(args.model_name_or_path, **options)\n    args.resolved_model_revision = getattr(model.config, \'_commit_hash\', None)\n    model.eval()\n    return model\n\n\ndef select_examples(raw, args):\n    count = max(1, int(len(raw[\'test\']) * args.data_prop)) if raw[\'test\'] else 0\n    if args.max_samples:\n        count = min(count, args.max_samples)\n    raw[\'test\'] = raw[\'test\'][:count]\n    if not raw[\'test\']:\n        raise ValueError(\'Test split is empty\')\n    if args.k_shot > len(raw[\'demo\']):\n        raise ValueError(f\'k_shot={args.k_shot} exceeds demo pool ({len(raw["demo"])})\')\n    return raw\n\n\ndef configure_tokenizer(tokenizer, model, args, output_tokens):\n    tokenizer.padding_side = \'left\'\n    tokenizer.truncation_side = \'left\'  # Keep the final question and Label: suffix.\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    if tokenizer.pad_token_id is None:\n        raise ValueError(\'Tokenizer needs a pad or EOS token\')\n    limits = [getattr(tokenizer, \'model_max_length\', 10**9),\n              getattr(model.config, \'max_position_embeddings\', 10**9)]\n    context = min(int(v) for v in limits if v is not None)\n    available = context - output_tokens\n    if available < 1:\n        raise ValueError(\'Generation budget exceeds the model context window\')\n    if args.max_seq_length == -1:\n        args.max_seq_length = available\n    else:\n        args.max_seq_length = min(args.max_seq_length, available)\n\n\ndef limit_prompts(examples, tokenizer, max_length):\n    """Truncate demonstrations first; reject loss of the actual test sentence."""\n    truncated = 0\n    for example in examples:\n        text = example[\'prompts\']\n        # The final prompt suffix contains the complete test input.\n        suffix = f"Sentence: {example[\'sentence\']}\\nLabel:"\n        if len(tokenizer(suffix, add_special_tokens=False)[\'input_ids\']) + tokenizer.num_special_tokens_to_add() > max_length:\n            raise ValueError(\'Test sentence exceeds max_seq_length. Increase the limit; refusing to silently drop test text.\')\n        ids = tokenizer(text, add_special_tokens=True)[\'input_ids\']\n        if len(ids) > max_length:\n            truncated += 1\n    if truncated:\n        warnings.warn(f\'{truncated}/{len(examples)} prompts exceed {max_length} tokens. Left truncation removes instructions/demos; results differ from untruncated evaluation.\')\n    return truncated\n\n\ndef run_metadata(args, raw, truncated):\n    from importlib.metadata import version\n    settings = {k: getattr(args, k) for k in (\n        \'model_name_or_path\', \'k_shot\', \'max_seq_length\', \'max_samples\', \'data_prop\',\n        \'demo_split\', \'resolved_dtype\', \'load_in_4bit\', \'max_new_tokens\', \'eval_batch_size\')}\n    fingerprint = hashlib.sha256(json.dumps(raw, sort_keys=True).encode()).hexdigest()\n    source_files = [Path(__file__), Path(__file__).with_name(\'fsa.py\'),\n                    Path(__file__).with_name(\'icl_multifaced.py\'), Path(__file__).with_name(\'utils.py\')]\n    source_files += sorted(Path(__file__).parent.glob(\'parse_utils/*.py\'))\n    source_hash = hashlib.sha256(b\'\'.join(p.read_bytes() for p in source_files)).hexdigest()\n    return dict(experiment=args.subname, dataset=args.dataset, seed=args.seed,\n                settings=settings, data_sha256=fingerprint, num_samples=len(raw[\'test\']),\n                source_sha256=source_hash, model_revision=args.resolved_model_revision,\n                truncated_prompts=truncated,\n                versions={p: version(p) for p in (\'torch\', \'transformers\', \'accelerate\', \'numpy\', \'scikit-learn\')})\n\n\nclass PredictionCache:\n    """Atomic batch checkpoints; validate data/settings before reusing predictions."""\n    def __init__(self, path, metadata):\n        self.path = Path(path) if path else None\n        self.metadata = metadata\n        self.entries = {}\n        if self.path and self.path.exists():\n            saved = json.loads(self.path.read_text(encoding=\'utf-8\'))\n            if saved[\'metadata\'] != metadata:\n                raise ValueError(\'Checkpoint settings/data/environment differ. Choose a new resume_file or restore the original configuration.\')\n            self.entries = saved[\'entries\']\n            print(f\'Resuming {len(self.entries)} cached predictions\')\n\n    @staticmethod\n    def key(example):\n        return hashlib.sha256(example[\'prompts\'].encode()).hexdigest()\n\n    def get(self, example):\n        return self.entries.get(self.key(example))\n\n    def put(self, example, value):\n        self.entries[self.key(example)] = value\n\n    def flush(self):\n        if self.path:\n            self.path.parent.mkdir(parents=True, exist_ok=True)\n            temp = self.path.with_suffix(self.path.suffix + \'.tmp\')\n            temp.write_text(json.dumps({\'metadata\': self.metadata, \'entries\': self.entries}), encoding=\'utf-8\')\n            os.replace(temp, self.path)\n', 'check_notebook_environment.py': '"""Validate the evaluator\'s dependency closure without importing unrelated host apps."""\nimport argparse\nimport importlib\nfrom importlib.metadata import PackageNotFoundError, distribution, version\nimport json\nfrom pathlib import Path\nimport platform\nimport sys\n\nfrom packaging.requirements import Requirement\n\n\ndef check(requirements_path, four_bit=False, require_gpu=False):\n    requirements = [Requirement(line.strip()) for line in Path(requirements_path).read_text().splitlines()\n                    if line.strip() and not line.lstrip().startswith(\'#\')]\n    requirements.extend([Requirement(\'torch>=2.2,<3\'), Requirement(\'packaging>=23\')])\n    if four_bit:\n        requirements.append(Requirement(\'bitsandbytes==0.45.5\'))\n    pending = [(r, \'notebook requirements\') for r in requirements]\n    visited, errors = set(), []\n    while pending:\n        requirement, parent = pending.pop()\n        if requirement.marker and not requirement.marker.evaluate({\'extra\': \'\'}):\n            continue\n        try:\n            dist = distribution(requirement.name)\n        except PackageNotFoundError:\n            errors.append(f\'{parent}: missing {requirement}\')\n            continue\n        if requirement.specifier and not requirement.specifier.contains(dist.version, prereleases=True):\n            errors.append(f\'{parent}: needs {requirement}, found {dist.version}\')\n        key = dist.metadata[\'Name\'].lower().replace(\'_\', \'-\')\n        if key in visited:\n            continue\n        visited.add(key)\n        pending.extend((Requirement(r), dist.metadata[\'Name\']) for r in dist.requires or [])\n    if errors:\n        raise RuntimeError(\'Dependency conflicts:\\n\' + \'\\n\'.join(sorted(set(errors))))\n    for module in (\'torch\', \'transformers\', \'accelerate\', \'numpy\', \'scipy\', \'sklearn\', \'nltk\', \'sentencepiece\',\n                   \'fsa\', \'icl_multifaced\'):\n        importlib.import_module(module)\n    import torch\n    from transformers import GPT2Config, GPT2LMHeadModel\n    if require_gpu and not torch.cuda.is_available():\n        raise RuntimeError(\'No CUDA GPU. Enable a GPU accelerator, then rerun.\')\n    device = \'cuda:0\' if torch.cuda.is_available() else \'cpu\'\n    dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.startswith(\'cuda\') else torch.float32\n    model = GPT2LMHeadModel(GPT2Config(vocab_size=32, n_positions=32, n_embd=16,\n                                     n_layer=1, n_head=2, bos_token_id=1, eos_token_id=2, pad_token_id=0))\n    model = model.to(device=device, dtype=dtype).eval()\n    with torch.inference_mode():\n        inputs = torch.tensor([[1, 3, 4]], device=device)\n        logits = model(inputs, attention_mask=torch.ones_like(inputs)).logits\n        assert torch.isfinite(logits).all()\n        generated = model.generate(inputs, attention_mask=torch.ones_like(inputs), max_new_tokens=1, do_sample=False)\n        assert generated.shape[-1] == 4\n    if four_bit:\n        import bitsandbytes as bnb\n        layer = bnb.nn.Linear4bit(64, 64, bias=False, compute_dtype=dtype, quant_type=\'nf4\').to(device)\n        with torch.inference_mode():\n            value = layer(torch.randn(2, 64, device=device, dtype=dtype))\n            assert torch.isfinite(value).all()\n        del layer\n    report = {\n        \'python\': sys.version, \'platform\': platform.platform(), \'executable\': sys.executable,\n        \'torch\': version(\'torch\'), \'cuda_build\': torch.version.cuda,\n        \'gpu\': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n        \'dtype\': str(dtype), \'four_bit_tested\': four_bit,\n        \'packages\': {name: version(name) for name in sorted(visited)},\n        \'dependency_check\': \'passed\', \'tiny_model_forward_and_generation\': \'passed\',\n    }\n    print(json.dumps(report, indent=2))\n    del model, logits, generated\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--requirements\', default=str(Path(__file__).with_name(\'requirements-notebook.txt\')))\n    parser.add_argument(\'--four-bit\', action=\'store_true\')\n    parser.add_argument(\'--require-gpu\', action=\'store_true\')\n    parser.add_argument(\'--report\', default=\'environment-report.json\')\n    args = parser.parse_args()\n    report = check(args.requirements, args.four_bit, args.require_gpu)\n    Path(args.report).write_text(json.dumps(report, indent=2))\n', 'requirements-notebook.txt': '# Use the CUDA-enabled torch supplied by Kaggle/Colab; do not replace it here.\n# Targets Python 3.10-3.12. Dependency resolution/imports are checked on the live runtime.\npackaging==25.0\ntransformers==4.44.2\ntokenizers==0.19.1\nhuggingface-hub==0.25.2\naccelerate==0.34.2\nnumpy>=1.26.4,<2\nscipy>=1.11.4,<1.15\nscikit-learn>=1.4.2,<1.6\nnltk==3.9.1\ntqdm>=4.66,<5\nsafetensors>=0.4.5,<0.6\nsentencepiece>=0.2,<0.3\n# Optional 4-bit on compatible CUDA hardware: pip install bitsandbytes==0.45.5\n', 'model_name.json': '{\n  "Qwen2.5-1.5B-sentiment-distillation-v1": "./model_weights/Qwen2.5-1.5B-sentiment-distillation-v1",\n  "llama-3-1B-sentiment-distillation-v1": "./model_weights/llama-3-1B-sentiment-distillation-v1",\n  "llama-3-3B-sentiment-distillation-v1": "./model_weights/llama-3-3B-sentiment-distillation-v1"\n}\n', 'parse_utils/acsa.py': 'from utils import append_new_line, save_json\nimport os, time, json\nfrom datetime import datetime\nimport ast\nimport re\n\nclass F1Measure:\n    def __init__(self):\n        self.pred_list = []  # List to store predictions\n        self.true_list = []  # List to store ground truths\n\n    def add_predictions(self, idx, preds):\n        """Adds a batch of predictions for a specific index."""\n        self.pred_list.extend((idx, pred) for pred in preds)\n\n    def add_ground_truths(self, idx, trues):\n        """Adds a batch of ground truths for a specific index."""\n        self.true_list.extend((idx, true) for true in trues)\n\n    def report(self):\n        """Calculates and returns the F1 score."""\n        self.f1, self.precision, self.recall = self.calculate_f1()\n        return self.f1\n\n    def __getitem__(self, key):\n        """Allows retrieval of attributes like a dictionary."""\n        if hasattr(self, key):\n            return getattr(self, key)\n        raise AttributeError(f"{key} is not a valid attribute of F1Measure.")\n\n    def calculate_f1(self):\n        """Calculates F1 score along with precision and recall."""\n        n_tp = sum(pred in self.true_list for pred in self.pred_list)\n        precision = n_tp / len(self.pred_list) if self.pred_list else 1\n\n        n_tp = sum(true in self.pred_list for true in self.true_list)\n        recall = n_tp / len(self.true_list) if self.true_list else 1\n\n        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0\n\n        return f1, precision, recall\n\n\ndef parse_category_polarity_for_true(acsa_seq, sentence):\n    category_polarity_list = []\n    valid_flag = True\n\n\n    for sub_seq in acsa_seq:\n\n        category_polarity_list.append(tuple(sub_seq))\n\n    return category_polarity_list, valid_flag\n\n# TODO: invalid_string\ndef parse_aspect_polarity(atsa_seq):\n    aspect_polarity_list = []\n\n    def parse_seq(category_polarity):\n        if type(category_polarity) in (tuple, list):\n            category, polarity = category_polarity\n        else:\n            category, polarity = re.split(r\',|-|\\|\', category_polarity)\n\n        category = category.strip()\n        polarity = polarity.strip()\n\n        if polarity not in (\'positive\', \'neutral\', \'negative\'):\n            return False\n\n        return category, polarity\n\n    atsa_seq = atsa_seq.replace(\'```python\', \'\')\n    atsa_seq = atsa_seq.replace(\'```\', \'\')\n    atsa_seq = atsa_seq.strip()\n    atsa_seq = atsa_seq.replace(\'\\n]\', \']\')\n    atsa_seq = atsa_seq.replace(\'[\\n\', \'[\')\n    atsa_seq = atsa_seq.split(\'\\n\')[-1]\n    atsa_seq = atsa_seq.replace(\'Label:\', \'\')\n    atsa_seq = atsa_seq.strip()\n    # atsa_seq = atsa_seq.replace(\'\\n\', \',\')\n    # atsa_seq = atsa_seq.replace(\',,\', \',\')\n    # print(\'string:\', atsa_seq)\n    try:\n        # aspect_polarity_list = eval(atsa_seq)\n        aspect_polarity_list = ast.literal_eval(atsa_seq)\n    except:\n        print(f"eval-error \'{atsa_seq}\'", )\n        try: \n            \n            last_paren_index = atsa_seq.rfind(")")\n\n            if last_paren_index != -1:\n                atsa_seq = atsa_seq[:last_paren_index+1] + "]"\n            else:\n                atsa_seq = atsa_seq + "]"  # 兜底处理\n\n            # 解析为 Python list\n            atsa_seq = ast.literal_eval(atsa_seq)\n            atsa_seq = set(atsa_seq)\n            atsa_seq = [tuple(item) for item in atsa_seq]\n            return atsa_seq[:3]\n        except:\n            return []\n\n    if type(aspect_polarity_list) not in (list, tuple):\n        print(\'type not list\', atsa_seq)\n        return []\n\n    if len(aspect_polarity_list) == 2 and type(aspect_polarity_list[0]) is str:\n        aspect_polarity_list = [aspect_polarity_list]\n\n    # 合并三级列表\n    # if len(aspect_polarity_list) > 0 and len(aspect_polarity_list[0]) > 0 and type(aspect_polarity_list[0][0]) in (list, tuple):\n    #     # _list = []\n    #     # for item in aspect_polarity_list:\n    #     #     _list.extend(item)\n    #     # aspect_polarity_list = _list\n    #     aspect_polarity_list = aspect_polarity_list[-1]\n\n    try:\n        aspect_polarity_list = [\n            parse_seq(aspect_polarity)\n            for aspect_polarity in aspect_polarity_list if aspect_polarity\n        ]\n    except Exception as e:\n        print(e)\n        print(\'string:\', atsa_seq)\n        print(\'parsed:\', aspect_polarity_list)\n\n    return aspect_polarity_list\n\n\n\n\nclass Result_acsa:\n    def __init__(self, data):\n        self.data = data\n\n    def __ge__(self, other):\n        return self.monitor >= other.monitor\n\n    def __gt__(self, other):\n        return self.monitor > other.monitor\n\n    @classmethod\n    def parse_from(cls, outputs):\n        data = {}\n\n        ID = 0\n        for example, prediction in outputs:\n\n            sentence = example[\'sentence\']\n\n\n            category_polarity_list_true = parse_category_polarity_for_true(example[\'label_seq\'], sentence)[0]\n            category_polarity_list_pred = parse_aspect_polarity(prediction)\n#                \'original_output_of_model\':original_prediction,\n\n\n            print(\'sentence:\',sentence)\n            print(\'original prediction:\',prediction)\n\n            print(\'true: \',category_polarity_list_true)\n            print(\'pred: \',category_polarity_list_pred)\n            print(\'-\'*100)\n\n\n            data[ID] = {\n                \'ID\': example.get(\'ID\', ID),\n                \'sentence\': sentence,\n                \'prompts\':example[\'prompts\'],\n                \'label_seq\':example[\'label_seq\'],\n                \'golden_label_parsed\': category_polarity_list_true,\n                \'prediction_parsed\': category_polarity_list_pred,\n                \'dataset\': example[\'dataset\']\n            }\n            ID += 1\n\n        return cls(data)\n\n    def cal_metric(self):\n        f1 = F1Measure()\n\n        for ID in self.data:\n            example = self.data[ID]\n            f1.add_ground_truths(ID, example[\'golden_label_parsed\'])\n            f1.add_predictions(ID, example[\'prediction_parsed\'])\n\n        f1.report()\n\n        self.detailed_metrics = {\n            \'f1\': f1[\'f1\'],\n            \'recall\': f1[\'recall\'],\n            \'precision\': f1[\'precision\'],\n        }\n\n        self.monitor = self.detailed_metrics[\'f1\']\n\n    def save_prediction(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr, experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        file_name = os.path.join(output_dir, \'result\', f\'{experiment_name}_{dataset}_{subname}_seed:{seed}_rank:{rank}_alpha:{alpha}_target_module:{target_module}_dropout:{dropout}_lr:{lr}.json\')\n\n        print(\'save prediction to\', file_name)\n        save_json(\n            {\n                \'data\': self.data,\n                \'meta\': (model_name_or_path, subname, dataset, seed, lr, now)\n            },\n            file_name\n        )\n\n\n\n    def save_metric(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr,experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        performance_file_name = os.path.join(output_dir, \'performance\' , now, \'performance.txt\')\n\n        print(\'save performace to\', performance_file_name)\n        append_new_line(performance_file_name, json.dumps({\n            \'experiment_name\':experiment_name,\n            \'time\': time.strftime(\'%Y-%m-%d %H_%M_%S\', time.localtime()),\n            \'model_name_or_path\': model_name_or_path,\n            \'subname\': subname,\n            \'dataset\': dataset,\n            \'seed\': seed,\n            \'lr\': lr,\n            \'rank\':rank,\n            \'alpha\': alpha,\n            \'target_module\': target_module,\n            \'dropout\': dropout,\n            \'metric\': self.detailed_metrics[\'f1\']\n        }))\n\n    def report(self):\n        for metric_names in ((\'precision\', \'recall\', \'f1\'),):\n            for metric_name in metric_names:\n                value = self.detailed_metrics[metric_name] if metric_name in self.detailed_metrics else 0\n                print(f\'{metric_name}: {value:.4f}\', end=\' | \')\n            print()\n', 'parse_utils/asqp.py': 'from utils import append_new_line, save_json\nimport os, time, json\nfrom datetime import datetime\nimport ast\n\nclass F1Measure:\n    def __init__(self):\n        self.pred_list = []  # List to store predictions\n        self.true_list = []  # List to store ground truths\n\n    def add_predictions(self, idx, preds):\n        """Adds a batch of predictions for a specific index."""\n        self.pred_list.extend((idx, pred) for pred in preds)\n\n    def add_ground_truths(self, idx, trues):\n        """Adds a batch of ground truths for a specific index."""\n        self.true_list.extend((idx, true) for true in trues)\n\n    # def add_predictions(self, idx, preds):\n    #     """Adds a batch of predictions for a specific index."""\n    #     self.pred_list.append(preds)\n\n    # def add_ground_truths(self, idx, trues):\n    #     """Adds a batch of ground truths for a specific index."""\n    #     self.true_list.append(trues)\n\n\n    def report(self):\n        """Calculates and returns the F1 score."""\n        self.f1, self.precision, self.recall = self.calculate_f1()\n        return self.f1\n\n    def __getitem__(self, key):\n        """Allows retrieval of attributes like a dictionary."""\n        if hasattr(self, key):\n            return getattr(self, key)\n        raise AttributeError(f"{key} is not a valid attribute of F1Measure.")\n\n    def calculate_f1(self):\n        """Calculates F1 score along with precision and recall."""\n\n        # print(\'pred_list:\', self.pred_list)\n        # print(\'true_list:\', self.true_list)\n\n        # for item in self.true_list:\n        #     index, true = item\n        #     print(\'true:\', )\n        #     print(true)\n        #     print(\'pred:\')\n        #     for pred in self.pred_list:\n        #         if index == pred[0]:\n        #             print(pred[1])\n        #     print(\'-\'*100)\n\n        n_tp = sum(pred in self.true_list for pred in self.pred_list)\n        precision = n_tp / len(self.pred_list) if self.pred_list else 1\n\n        n_tp = sum(true in self.pred_list for true in self.true_list)\n        recall = n_tp / len(self.true_list) if self.true_list else 1\n\n        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0\n\n        # micro_f1 = self.process_tuple_f1(self.true_list, self.pred_list)\n\n        return f1, precision, recall\n\n\n    def process_tuple_f1(self, labels, predictions, verbose=False):\n        tp, fp, fn = 0, 0, 0\n        epsilon = 1e-7\n        for i in range(len(labels)):\n            gold = set(labels[i])\n            try:\n                pred = set(predictions[i])\n            except Exception:\n                pred = set()\n\n            print(\'gold:\', gold)\n            print(\'pred:\', pred)\n            print(\'-\'*100)\n            tp += len(gold.intersection(pred))\n            fp += len(pred.difference(gold))\n            fn += len(gold.difference(pred))\n        if verbose:\n            print(\'-\'*100)\n            print(gold, pred)\n        precision = tp / (tp + fp + epsilon)\n        recall = tp / (tp + fn + epsilon)\n        micro_f1 = 2 * (precision * recall) / (precision + recall + epsilon)\n        return micro_f1\n\n\ndef parse_quards_for_true(quards_seq, sentence):\n    quards_tuple = []\n    for item in quards_seq:\n        quards_tuple.append(tuple(item))\n    print(\'true:\', quards_seq)\n    return quards_tuple\n\n\ndef parse_quards_for_predict(quards_seq, sentence):\n\n\n    quards_seq = quards_seq.replace(\'```python\', \'\')\n    quards_seq = quards_seq.replace(\'```\', \'\')\n    quards_seq = quards_seq.strip()\n    quards_seq = quards_seq.replace(\'\\n]\', \']\')\n    quards_seq = quards_seq.replace(\'[\\n\', \'[\')\n    quards_seq = quards_seq.split(\'\\n\')[-1]\n    quards_seq = quards_seq.strip()\n\n\n    valid_flag = True\n    print("before parsed :", quards_seq)\n\n    try:\n        quards_seq = ast.literal_eval(quards_seq)\n        \n        \n        quards_seq = set(quards_seq)\n\n        quards_seq = [tuple(item) for item in quards_seq]\n        print("Parsed1 List:", quards_seq)\n    except:\n        try: \n            \n            last_paren_index = quards_seq.rfind(")")\n\n            if last_paren_index != -1:\n                quards_seq = quards_seq[:last_paren_index+1] + "]"\n            else:\n                quards_seq = quards_seq + "]"  # 兜底处理\n\n            # 解析为 Python list\n            quards_seq = ast.literal_eval(quards_seq)\n            quards_seq = set(quards_seq)\n            quards_seq = [tuple(item) for item in quards_seq][:3]\n            print("Parsed List2:", quards_seq)\n        except:\n                quards_seq = []\n\n\n    def parse_seq(item):\n        \n        if len(item) != 4:\n            return False\n\n        category, aspect , opinion, sentiment = item\n        category = category.strip()\n        aspect = aspect.strip()\n        opinion = opinion.strip()\n        sentiment = sentiment.strip()\n\n\n        return category, aspect, opinion, sentiment\n\n    quards_list = []\n\n    for sub_seq in quards_seq:\n        quards = parse_seq(sub_seq)\n        if not quards:\n            valid_flag = False\n        else:\n            quards_list.append(quards)\n\n    print(\'-\'*100)\n    print(\'final_list:\', quards_list)\n    print(\'-\'*100)\n\n    return quards_seq, quards_list, valid_flag\n\n\nclass Result_asqp:\n    def __init__(self, data):\n        self.data = data\n\n    def __ge__(self, other):\n        return self.monitor >= other.monitor\n\n    def __gt__(self, other):\n        return self.monitor > other.monitor\n\n    @classmethod\n    def parse_from(cls, outputs):\n\n        data = {}\n\n        ID = 0\n        for example, prediction  in outputs:\n\n\n            \n            sentence = example[\'sentence\']\n\n            asqp_list_true = parse_quards_for_true(example[\'label_seq\'], sentence)\n            original_prediction, asqp_list_pred = parse_quards_for_predict(prediction,\n                                                                                               sentence)[:2]\n\n            # print(asqp_list_pred)\n            # print(asqp_list_true)\n\n            data[ID] = {\n                \'ID\': example.get(\'ID\', ID),\n                \'sentence\': sentence,\n                \'prompts\':example[\'prompts\'],\n                \'label_seq\':example[\'label_seq\'],\n                \'golden_label_parsed\': asqp_list_true,\n                \'prediction_parsed\': asqp_list_pred,\n                \'original_output_of_model\': original_prediction,\n                \'dataset\': example[\'dataset\']\n                \n            }\n            ID += 1\n\n        return cls(data)\n\n    def cal_metric(self):\n        f1 = F1Measure()\n\n        for ID in self.data:\n            example = self.data[ID]\n            f1.add_ground_truths(ID, example[\'golden_label_parsed\'])\n            f1.add_predictions(ID, example[\'prediction_parsed\'])\n\n            \n            # print(\'a\'*100)\n            # print(\'golden_label:\')\n            # print(example[\'golden_label\'])\n            \n            # print(\'a\' * 100)\n            # print(\'prediction:\')\n            # print(example[\'prediction\'])\n\n\n        f1.report()\n\n        self.detailed_metrics = {\n            \'f1\': f1[\'f1\'],\n            \'recall\': f1[\'recall\'],\n            \'precision\': f1[\'precision\'],\n        }\n\n        self.monitor = self.detailed_metrics[\'f1\']\n\n    def save_prediction(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr, experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        file_name = os.path.join(output_dir, \'result\', f\'{experiment_name}_{dataset}_{subname}_seed:{seed}_rank:{rank}_alpha:{alpha}_target_module:{target_module}_dropout:{dropout}_lr:{lr}.json\')\n\n        print(\'save prediction to\', file_name)\n        save_json(\n            {\n                \'data\': self.data,\n                \'meta\': (model_name_or_path, subname, dataset, seed, lr, now)\n            },\n            file_name\n        )\n\n\n\n    def save_metric(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr,experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        performance_file_name = os.path.join(output_dir, \'performance\', \'performance.txt\')\n\n        print(\'save performace to\', performance_file_name)\n        append_new_line(performance_file_name, json.dumps({\n            \'experiment_name\':experiment_name,\n            \'metric\': self.detailed_metrics[\'f1\'],\n            \'dataset\': dataset,\n            \'subname\': subname,\n            \'seed\': seed,\n            \'time\': time.strftime(\'%Y-%m-%d %H_%M_%S\', time.localtime()),\n            \'model_name_or_path\': model_name_or_path,\n            \'lr\': lr,\n            \'rank\':rank,\n            \'alpha\': alpha,\n            \'target_module\': target_module,\n            \'dropout\': dropout,\n            \n        }))\n\n    def report(self):\n        for metric_names in ((\'precision\', \'recall\', \'f1\'),):\n            for metric_name in metric_names:\n                value = self.detailed_metrics[metric_name] if metric_name in self.detailed_metrics else 0\n                print(f\'{metric_name}: {value:.4f}\', end=\' | \')\n            print()\n', 'parse_utils/atsa.py': 'from utils import append_new_line, save_json\nimport os, time, json\nfrom datetime import datetime\nimport re\nimport ast\n\nclass F1Measure:\n    def __init__(self):\n        self.pred_list = []  # List to store predictions\n        self.true_list = []  # List to store ground truths\n\n    def add_predictions(self, idx, preds):\n        """Adds a batch of predictions for a specific index."""\n        self.pred_list.extend((idx, pred) for pred in preds)\n\n    def add_ground_truths(self, idx, trues):\n        """Adds a batch of ground truths for a specific index."""\n        self.true_list.extend((idx, true) for true in trues)\n\n    def report(self):\n        """Calculates and returns the F1 score."""\n        self.f1, self.precision, self.recall = self.calculate_f1()\n        return self.f1\n\n    def __getitem__(self, key):\n        """Allows retrieval of attributes like a dictionary."""\n        if hasattr(self, key):\n            return getattr(self, key)\n        raise AttributeError(f"{key} is not a valid attribute of F1Measure.")\n\n    def calculate_f1(self):\n        """Calculates F1 score along with precision and recall."""\n        n_tp = sum(pred in self.true_list for pred in self.pred_list)\n        precision = n_tp / len(self.pred_list) if self.pred_list else 1\n\n        n_tp = sum(true in self.pred_list for true in self.true_list)\n        recall = n_tp / len(self.true_list) if self.true_list else 1\n\n        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0\n\n        return f1, precision, recall\n\n\n# TODO: invalid_string\ndef parse_aspect_polarity(atsa_seq, sentence):\n    aspect_polarity_list = []\n\n    def parse_seq(aspect_polarity):\n        if type(aspect_polarity) in (tuple, list):\n            aspect, polarity = aspect_polarity\n        else:\n            # aspect, polarity = aspect_polarity.split(\',\')\n            aspect, polarity = re.split(r\',|-|\\|\', aspect_polarity)\n\n        aspect = aspect.strip()\n        polarity = polarity.strip()\n\n        if aspect not in sentence:\n            return False \n\n        if polarity not in (\'positive\', \'neutral\', \'negative\', \'conflict\'):\n            return False\n\n        return aspect, polarity\n\n    atsa_seq = atsa_seq.replace(\'```python\', \'\')\n    atsa_seq = atsa_seq.replace(\'```\', \'\')\n    atsa_seq = atsa_seq.strip()\n    atsa_seq = atsa_seq.replace(\'\\n]\', \']\')\n    atsa_seq = atsa_seq.replace(\'[\\n\', \'[\')\n    atsa_seq = atsa_seq.split(\'\\n\')[-1]\n    atsa_seq = atsa_seq.replace(\'Label:\', \'\')\n    atsa_seq = atsa_seq.strip()\n    # atsa_seq = atsa_seq.replace(\'\\n\', \',\')\n    # atsa_seq = atsa_seq.replace(\',,\', \',\')\n    # print(\'string:\', atsa_seq)\n    try:\n        # aspect_polarity_list = eval(atsa_seq)\n        aspect_polarity_list = ast.literal_eval(atsa_seq)\n    except:\n        print(f"eval-error \'{atsa_seq}\'", )\n        return []\n\n    if type(aspect_polarity_list) not in (list, tuple):\n        print(\'type not list\', atsa_seq)\n        return []\n\n    if len(aspect_polarity_list) == 2 and type(aspect_polarity_list[0]) is str:\n        aspect_polarity_list = [aspect_polarity_list]\n\n    # 合并三级列表\n\n    try:\n        if len(aspect_polarity_list) > 0 and len(aspect_polarity_list[0]) > 0 and type(aspect_polarity_list[0][0]) in (list, tuple):\n            # _list = []\n            # for item in aspect_polarity_list:\n            #     _list.extend(item)\n            # aspect_polarity_list = _list\n            aspect_polarity_list = aspect_polarity_list[-1]\n    except:\n        return []\n\n    try:\n        aspect_polarity_list = [\n            parse_seq(aspect_polarity)\n            for aspect_polarity in aspect_polarity_list if aspect_polarity\n        ]\n    except:\n        print(\'string:\', atsa_seq)\n        print(\'parsed:\', aspect_polarity_list)\n\n    return aspect_polarity_list\n\n\ndef parse_aspect_polarity_for_true(atsa_seq, sentence):\n    aspect_polarity_list = []\n\n    valid_flag = True\n\n    for sub_seq in atsa_seq:\n\n        aspect_polarity_list.append(tuple(sub_seq))\n\n    # print(aspect_polarity_list)\n\n    return aspect_polarity_list, valid_flag\n\n\nclass Result_atsa:\n    def __init__(self, data):\n        self.data = data \n\n    def __ge__(self, other):\n        return self.monitor >= other.monitor\n\n    def __gt__(self, other):\n        return self.monitor >  other.monitor\n\n    @classmethod\n    def parse_from(cls, outputs):\n\n\n\n        data = {}\n\n        ID = 0\n        for example, prediction in outputs:\n\n\n            sentence = example[\'sentence\']\n            print(\'sentence:\',sentence)\n            print(\'original prediction:\',prediction)\n            aspect_polarity_list_true = parse_aspect_polarity_for_true(example[\'label_seq\'], sentence)[0]\n            aspect_polarity_list_pred = parse_aspect_polarity(prediction, sentence)\n#                \'original_output_of_model\':original_prediction,\n\n\n            print(aspect_polarity_list_true)\n            print(aspect_polarity_list_pred)\n            print(\'-\'*100)\n\n            data[ID] = {\n                \'ID\': example.get(\'ID\', ID),\n                \'sentence\': sentence,\n                \'prompts\':example[\'prompts\'],\n                \'label_seq\':example[\'label_seq\'],\n                \'golden_label_parsed\': aspect_polarity_list_true,\n                \'prediction_parsed\': aspect_polarity_list_pred,\n                \'dataset\': example[\'dataset\']\n            }\n            ID += 1\n\n        return cls(data)\n\n    def cal_metric(self):\n        f1 = F1Measure()\n\n        for ID in self.data:\n            example = self.data[ID]\n            f1.add_ground_truths(ID, example[\'golden_label_parsed\'])\n            f1.add_predictions(ID, example[\'prediction_parsed\'])\n\n        f1.report()\n\n        self.detailed_metrics = {\n            \'f1\': f1[\'f1\'],\n            \'recall\': f1[\'recall\'],\n            \'precision\': f1[\'precision\'],\n        }\n\n        self.monitor = self.detailed_metrics[\'f1\']\n\n    def save_prediction(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr, experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        file_name = os.path.join(output_dir, \'result\', f\'{experiment_name}_{dataset}_{subname}_seed:{seed}_rank:{rank}_alpha:{alpha}_target_module:{target_module}_dropout:{dropout}_lr:{lr}.json\')\n\n        print(\'save prediction to\', file_name)\n        save_json(\n            {\n                \'data\': self.data,\n                \'meta\': (model_name_or_path, subname, dataset, seed, lr, now)\n            },\n            file_name\n        )\n\n\n\n    def save_metric(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr,experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        performance_file_name = os.path.join(output_dir, \'performance\' , now, \'performance.txt\')\n\n        print(\'save performace to\', performance_file_name)\n        append_new_line(performance_file_name, json.dumps({\n            \'experiment_name\':experiment_name,\n            \'time\': time.strftime(\'%Y-%m-%d %H_%M_%S\', time.localtime()),\n            \'model_name_or_path\': model_name_or_path,\n            \'subname\': subname,\n            \'dataset\': dataset,\n            \'seed\': seed,\n            \'lr\': lr,\n            \'rank\':rank,\n            \'alpha\': alpha,\n            \'target_module\': target_module,\n            \'dropout\': dropout,\n            \'metric\': self.detailed_metrics[\'f1\']\n        }))\n\n    def report(self):\n        for metric_names in ((\'precision\', \'recall\', \'f1\'),):\n            for metric_name in metric_names:\n                value = self.detailed_metrics[metric_name] if metric_name in self.detailed_metrics else 0\n                print(f\'{metric_name}: {value:.4f}\', end=\' | \')\n            print()\n', 'parse_utils/coqe.py': 'from utils import append_new_line, save_json\nimport os, time, json\nfrom datetime import datetime\nimport ast\n\nclass F1Measure:\n    def __init__(self):\n        self.pred_list = []  # List to store predictions\n        self.true_list = []  # List to store ground truths\n\n    def add_predictions(self, idx, preds):\n        """Adds a batch of predictions for a specific index."""\n        self.pred_list.extend((idx, pred) for pred in preds)\n\n    def add_ground_truths(self, idx, trues):\n        """Adds a batch of ground truths for a specific index."""\n        self.true_list.extend((idx, true) for true in trues)\n\n    def report(self):\n        """Calculates and returns the F1 score."""\n        self.f1, self.precision, self.recall = self.calculate_f1()\n        return self.f1\n\n    def __getitem__(self, key):\n        """Allows retrieval of attributes like a dictionary."""\n        if hasattr(self, key):\n            return getattr(self, key)\n        raise AttributeError(f"{key} is not a valid attribute of F1Measure.")\n\n    def calculate_f1(self):\n        """Calculates F1 score along with precision and recall."""\n\n\n\n        n_tp = sum(pred in self.true_list for pred in self.pred_list)\n        precision = n_tp / len(self.pred_list) if self.pred_list else 1\n\n        n_tp = sum(true in self.pred_list for true in self.true_list)\n        recall = n_tp / len(self.true_list) if self.true_list else 1\n\n        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0\n\n        return f1, precision, recall\n\n\n\ndef parse_quintuple_for_true(quintuple_seq, sentence):\n\n    quintuple_tuple = []\n    for sub_seq in quintuple_seq:\n        new_sub_seq = []\n        for item in sub_seq:\n            new_sub_seq.append(item.lower())\n        quintuple_tuple.append(tuple(new_sub_seq))\n\n    print(\'true:\', quintuple_tuple)\n    return quintuple_tuple\n\n\ndef parse_quintuple_for_predict(quintuple_seq, sentence):\n\n\n    quintuple_seq = quintuple_seq.replace(\'```python\', \'\')\n    quintuple_seq = quintuple_seq.replace(\'```\', \'\')\n    quintuple_seq = quintuple_seq.strip()\n    quintuple_seq = quintuple_seq.replace(\'\\n]\', \']\')\n    quintuple_seq = quintuple_seq.replace(\'[\\n\', \'[\')\n    quintuple_seq = quintuple_seq.split(\'\\n\')[-1]\n    quintuple_seq = quintuple_seq.strip()\n\n\n    valid_flag = True\n    print("before parsed :", quintuple_seq)\n\n    try:\n        quintuple_seq = ast.literal_eval(quintuple_seq)\n        \n        \n        quintuple_seq = set(quintuple_seq)\n\n        quintuple_seq = [tuple(item) for item in quintuple_seq]\n        print("Parsed1 List:", quintuple_seq)\n    except:\n        try: \n            \n            last_paren_index = quintuple_seq.rfind(")")\n\n            if last_paren_index != -1:\n                quintuple_seq = quintuple_seq[:last_paren_index+1] + "]"\n            else:\n                quintuple_seq = quintuple_seq + "]"  # 兜底处理\n\n            # 解析为 Python list\n            quintuple_seq = ast.literal_eval(quintuple_seq)\n            quintuple_seq = set(quintuple_seq)\n            quintuple_seq = [tuple(item) for item in quintuple_seq]\n            print("Parsed List2:", quintuple_seq)\n        except:\n                quintuple_seq = []\n\n\n    def parse_seq(item):\n        if len(item) != 5:\n            return False\n\n\n        subject, object, aspect, opinion, preference = item\n        subject = subject.strip().lower()\n        object = object.strip().lower()\n        aspect = aspect.strip().lower()\n        opinion = opinion.strip().lower()\n        preference = preference.strip().lower()\n\n        return subject, object, aspect, opinion, preference \n\n\n    quintuple_list = []\n\n    for sub_seq in quintuple_seq:\n        quintuple = parse_seq(sub_seq)\n        if not quintuple:\n            valid_flag = False\n        else:\n            quintuple_list.append(quintuple)\n\n    print(\'-\'*100)\n    print(\'final_list:\', quintuple_list)\n    print(\'-\'*100)\n\n    return quintuple_seq, quintuple_list, valid_flag\n\n\nclass Result_coqe:\n    def __init__(self, data):\n        self.data = data\n\n    def __ge__(self, other):\n        return self.monitor >= other.monitor\n\n    def __gt__(self, other):\n        return self.monitor > other.monitor\n\n    @classmethod\n    def parse_from(cls, outputs):\n\n        data = {}\n\n        ID = 0\n\n        outputs_data = []\n        for example, prediction  in outputs:\n\n\n            sentence = example[\'sentence\']\n\n            coqe_list_true = parse_quintuple_for_true(example[\'label_seq\'], sentence)\n\n            # if coqe_list_true == [(\'null\', \'null\', \'null\', \'null\', \'null\')]:\n            #     print(\'all null, skip\')\n            #     continue\n\n            original_prediction, coqe_list_pred = parse_quintuple_for_predict(prediction,\n                                                                                               sentence)[:2]\n\n            # print(coqe_list_pred)\n            print(\'coqe_list_true:\', coqe_list_true)\n            print(\'coqe_list_pred:\', coqe_list_pred)\n            print(\'sentence:\', sentence)\n            print(\'-\'*100)\n\n\n            data[ID] = {\n                \'ID\': example.get(\'ID\', ID),\n                \'sentence\': sentence,\n                \'prompts\':example[\'prompts\'],\n                \'label_seq\':example[\'label_seq\'],\n                \'golden_label_parsed\': coqe_list_true,\n                \'prediction_parsed\': coqe_list_pred,\n                \'original_output_of_model\': original_prediction,\n                \'dataset\': example[\'dataset\']\n            }\n            ID += 1\n\n            outputs_data.append({\n                \'sentence\': sentence,\n                \'golden_label_parsed\': coqe_list_true,\n                \'prediction_parsed\': coqe_list_pred,\n            })\n\n        with open(\'outputs_data.json\', \'w\') as f:\n            json.dump(outputs_data, f, indent=4)\n\n        return cls(data)\n\n    def cal_metric(self):\n        f1 = F1Measure()\n\n        for ID in self.data:\n            example = self.data[ID]\n            f1.add_ground_truths(ID, example[\'golden_label_parsed\'])\n            f1.add_predictions(ID, example[\'prediction_parsed\'])\n\n\n\n        f1.report()\n\n        self.detailed_metrics = {\n            \'f1\': f1[\'f1\'],\n            \'recall\': f1[\'recall\'],\n            \'precision\': f1[\'precision\'],\n        }\n\n        self.monitor = self.detailed_metrics[\'f1\']\n\n    def save_prediction(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr, experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        file_name = os.path.join(output_dir, \'result\', f\'{experiment_name}_{dataset}_{subname}_seed:{seed}_rank:{rank}_alpha:{alpha}_target_module:{target_module}_dropout:{dropout}_lr:{lr}.json\')\n\n        print(\'save prediction to\', file_name)\n        save_json(\n            {\n                \'data\': self.data,\n                \'meta\': (model_name_or_path, subname, dataset, seed, lr, now)\n            },\n            file_name\n        )\n\n\n\n    def save_metric(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr,experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        performance_file_name = os.path.join(output_dir, \'performance\', \'performance.txt\')\n\n        print(\'save performace to\', performance_file_name)\n        append_new_line(performance_file_name, json.dumps({\n            \'experiment_name\':experiment_name,\n            \'metric\': self.detailed_metrics[\'f1\'],\n            \'dataset\': dataset,\n            \'subname\': subname,\n            \'seed\': seed,\n            \'time\': time.strftime(\'%Y-%m-%d %H_%M_%S\', time.localtime()),\n            \'model_name_or_path\': model_name_or_path,\n            \'lr\': lr,\n            \'rank\':rank,\n            \'alpha\': alpha,\n            \'target_module\': target_module,\n            \'dropout\': dropout,\n            \n        }))\n\n    def report(self):\n        for metric_names in ((\'precision\', \'recall\', \'f1\'),):\n            for metric_name in metric_names:\n                value = self.detailed_metrics[metric_name] if metric_name in self.detailed_metrics else 0\n                print(f\'{metric_name}: {value:.4f}\', end=\' | \')\n            print()\n', 'parse_utils/multifaced.py': 'from utils import append_new_line, save_json\nimport os, time, json\nfrom datetime import datetime\nfrom sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score\nfrom collections import Counter\nclass F1Measure:\n    def __init__(self):\n        self.pred_list = []  # List to store predictions\n        self.true_list = []  # List to store ground truths\n\n    def add_predictions(self, idx, preds):\n        """Adds a batch of predictions for a specific index."""\n        self.pred_list.extend((idx, pred) for pred in preds)\n\n    def add_ground_truths(self, idx, trues):\n        """Adds a batch of ground truths for a specific index."""\n        self.true_list.extend((idx, true) for true in trues)\n\n    def report(self):\n        """Calculates and returns the F1 score."""\n        self.f1, self.precision, self.recall = self.calculate_f1()\n        return self.f1\n\n    def __getitem__(self, key):\n        """Allows retrieval of attributes like a dictionary."""\n        if hasattr(self, key):\n            return getattr(self, key)\n        raise AttributeError(f"{key} is not a valid attribute of F1Measure.")\n\n    def calculate_f1(self):\n        """Calculates F1 score along with precision and recall."""\n        n_tp = sum(pred in self.true_list for pred in self.pred_list)\n        precision = n_tp / len(self.pred_list) if self.pred_list else 1\n\n        n_tp = sum(true in self.pred_list for true in self.true_list)\n        recall = n_tp / len(self.true_list) if self.true_list else 1\n\n        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0\n\n        return f1, precision, recall\n\n\n\n\nclass Result_multifaced:\n    def __init__(self, data):\n        self.data = data\n\n    def __ge__(self, other):\n        return self.monitor >= other.monitor\n\n    def __gt__(self, other):\n        return self.monitor > other.monitor\n\n    @classmethod\n    def parse_from(cls, outputs):\n\n        data = {}\n\n        ID = 0\n        for example, prediction in outputs:\n            sentence = example[\'sentence\']\n            # print(prediction)\n            extraction_index = prediction.find("Label:")\n            if extraction_index == -1:\n                prediction = \'\'\n            else:\n                prediction = prediction[(extraction_index + len("Label:")):]\n                prediction = prediction.strip()\n            \n            # print(\'-\'*100)\n            # print(prediction)\n            # print(\'-\'*100)\n\n\n            if example[\'dataset\'] == \'multifaced/irony18\':\n                if prediction != \'non-irony\' and prediction != \'irony\':\n                    prediction_processed = \'non-irony\'\n                else:\n                    prediction_processed = prediction\n            elif example[\'dataset\'] == \'multifaced/hateval\':\n                if prediction != \'non-hate\' and prediction != \'hate\':\n                    prediction_processed = \'non-hate\'\n                else:\n                    prediction_processed = prediction\n            elif example[\'dataset\'] == \'multifaced/compsent19\':\n                if prediction != \'better\' and prediction != \'worse\' and prediction != \'none\':\n                    prediction_processed = \'none\'\n                else:\n                    prediction_processed = prediction\n            elif example[\'dataset\'] == \'multifaced/offenseval\':\n                if prediction != \'non-offensive\' and prediction != \'offensive\':\n                    prediction_processed = \'non-offensive\'\n                else:\n                    prediction_processed = prediction\n            elif example[\'dataset\'] == \'multifaced/tweeteval\':\n                if prediction != \'anger\' and prediction != \'joy\' and prediction != \'optimism\' and prediction != \'sadness\':\n                    prediction_processed = \'anger\'\n                else:\n                    prediction_processed = prediction\n            elif example[\'dataset\'] == \'multifaced/pstance\':\n                if prediction != \'against\' and prediction != \'favor\':\n                    prediction_processed = \'favor\'\n                else:\n                    prediction_processed = prediction\n            elif example[\'dataset\'] == \'multifaced/intimacy\':\n                if prediction not in ["not intimate", "slightly intimate", "moderately intimate", "highly intimate"]:\n                    prediction_processed = \'not intimate\'\n                    print(\'error\')\n                else:\n                    prediction_processed = prediction\n            else:\n                prediction_processed = prediction\n                \n            data[ID] = {\n                \'ID\': example.get(\'ID\', ID),\n                \'sentence\': sentence,\n                \'prompts\': example[\'prompts\'],\n                \'golden_label\': example[\'label_seq\'],\n                \'prediction_processed\': prediction_processed,\n                \'original_output_of_model\': prediction,\n                \'dataset\': example[\'dataset\']\n            }\n            ID += 1\n\n        return cls(data)\n\n    def cal_metric(self):\n\n        ground_truths = []\n        predictions = []\n\n        dataset = self.data[0][\'dataset\']\n        for ID in self.data:\n            example = self.data[ID]\n            ground_truths.append(example[\'golden_label\'])\n            predictions.append(example[\'prediction_processed\'])\n        \n        # print(\'ground_truths:\', ground_truths)\n        print(Counter(ground_truths))\n\n        # print(\'predictions:\', predictions)\n        print(Counter(predictions))\n\n        macro_f1 = f1_score(ground_truths, predictions, average=\'macro\')\n        macro_precision = precision_score(ground_truths, predictions, average=\'macro\')\n        macro_recall = recall_score(ground_truths, predictions, average=\'macro\')\n        \n        \n    \n        self.detailed_metrics = {\n            \'f1\': macro_f1,\n            \'recall\': macro_precision,\n            \'precision\': macro_recall,\n        }\n\n        self.monitor = self.detailed_metrics[\'f1\']\n\n    def save_prediction(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr, experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        file_name = os.path.join(output_dir, \'result\', f\'{experiment_name}_{dataset}_{subname}_seed:{seed}_rank:{rank}_alpha:{alpha}_target_module:{target_module}_dropout:{dropout}_lr:{lr}.json\')\n\n        print(\'save prediction to\', file_name)\n        save_json(\n            {\n                \'data\': self.data,\n                \'meta\': (model_name_or_path, subname, dataset, seed, lr, now)\n            },\n            file_name\n        )\n\n\n\n    def save_metric(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr,experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        performance_file_name = os.path.join(output_dir, \'performance\', \'performance.txt\')\n\n        print(\'save performace to\', performance_file_name)\n        append_new_line(performance_file_name, json.dumps({\n            \'experiment_name\':experiment_name,\n            \'metric\': self.detailed_metrics[\'f1\'],\n            \'dataset\': dataset,\n            \'subname\': subname,\n            \'seed\': seed,\n            \'time\': time.strftime(\'%Y-%m-%d %H_%M_%S\', time.localtime()),\n            \'model_name_or_path\': model_name_or_path,\n            \'lr\': lr,\n            \'rank\':rank,\n            \'alpha\': alpha,\n            \'target_module\': target_module,\n            \'dropout\': dropout,\n            \n        }))\n\n    def report(self):\n        for metric_names in ((\'precision\', \'recall\', \'f1\'),):\n            for metric_name in metric_names:\n                value = self.detailed_metrics[metric_name] if metric_name in self.detailed_metrics else 0\n                print(f\'{metric_name}: {value:.4f}\', end=\' | \')\n            print()\n', 'parse_utils/ssa.py': 'from utils import append_new_line, save_json\nimport os, time, json\nfrom datetime import datetime\nimport ast\ndef parse_quards_for_true(true_seq):\n\n    print(\'true_seq:\', true_seq)\n    \n    true_tuple = []\n    for item in true_seq:\n        true_tuple.append(tuple(item))\n\n    print(\'true_tuple:\', true_tuple)\n    return true_tuple\n\n\ndef parse_quards_for_predict(quards_seq, sentence):\n\n\n    quards_seq = quards_seq.replace(\'```python\', \'\')\n    quards_seq = quards_seq.replace(\'```\', \'\')\n    quards_seq = quards_seq.strip()\n    quards_seq = quards_seq.replace(\'\\n]\', \']\')\n    quards_seq = quards_seq.replace(\'[\\n\', \'[\')\n    quards_seq = quards_seq.split(\'\\n\')[-1]\n    quards_seq = quards_seq.strip()\n\n\n\n    valid_flag = True\n    print("before parsed :", quards_seq)\n\n    try:\n        quards_seq = ast.literal_eval(quards_seq)\n        \n        \n        quards_seq = set(quards_seq)\n\n        quards_seq = [tuple(item) for item in quards_seq]\n        print("Parsed1 List:", quards_seq)\n    except:\n        try: \n            \n            last_paren_index = quards_seq.rfind(")")\n\n            if last_paren_index != -1:\n                quards_seq = quards_seq[:last_paren_index+1] + "]"\n            else:\n                quards_seq = quards_seq + "]"  # 兜底处理\n\n            # 解析为 Python list\n            quards_seq = ast.literal_eval(quards_seq)\n            quards_seq = set(quards_seq)\n            quards_seq = [tuple(item) for item in quards_seq]\n            print("Parsed List2:", quards_seq)\n        except:\n                quards_seq = []\n\n\n    def parse_seq(item):\n        if len(item) != 4:\n            return False\n\n        holder, target, expression, sentiment = item\n        holder = holder.strip() if type(holder) == str else \'NULL\'\n        target = target.strip()\n        expression = expression.strip()\n        sentiment = sentiment.strip()\n\n        if sentiment not in (\'positive\', \'neutral\', \'negative\',):\n            return False\n\n        return  holder, target, expression, sentiment\n\n    quards_list = []\n\n    for sub_seq in quards_seq:\n        quards = parse_seq(sub_seq)\n        if not quards:\n            valid_flag = False\n        else:\n            quards_list.append(quards)\n\n    print(\'-\'*100)\n    print(\'final_list:\', quards_list)\n    print(\'-\'*100)\n\n\n    return quards_seq, quards_list, valid_flag\n\n\nclass F1Measure:\n    def __init__(self):\n        self.pred_list = []  # List to store predictions\n        self.true_list = []  # List to store ground truths\n\n    def add_predictions(self, idx, preds):\n        """Adds a batch of predictions for a specific index."""\n        self.pred_list.extend((idx, pred) for pred in preds)\n\n    def add_ground_truths(self, idx, trues):\n        """Adds a batch of ground truths for a specific index."""\n        self.true_list.extend((idx, true) for true in trues)\n\n    def report(self):\n        """Calculates and returns the F1 score."""\n        self.f1, self.precision, self.recall = self.calculate_f1()\n        return self.f1\n\n    def __getitem__(self, key):\n        """Allows retrieval of attributes like a dictionary."""\n        if hasattr(self, key):\n            return getattr(self, key)\n        raise AttributeError(f"{key} is not a valid attribute of F1Measure.")\n\n    def calculate_f1(self):\n        """Calculates F1 score along with precision and recall."""\n        n_tp = sum(pred in self.true_list for pred in self.pred_list)\n        precision = n_tp / len(self.pred_list) if self.pred_list else 1\n\n        n_tp = sum(true in self.pred_list for true in self.true_list)\n        recall = n_tp / len(self.true_list) if self.true_list else 1\n\n        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0\n\n        return f1, precision, recall\n\nfrom nltk.tokenize.simple import SpaceTokenizer\n\ntk = SpaceTokenizer()\n\ndef convert_char_offsets_to_token_idxs(char_offsets, token_offsets):\n\n    token_idxs = []\n    #\n    for char_offset in char_offsets:\n        bidx, eidx = char_offset.split(":")\n        bidx, eidx = int(bidx), int(eidx)\n        intoken = False\n        for i, (b, e) in enumerate(token_offsets):\n            if b == bidx:\n                intoken = True\n            if intoken:\n                token_idxs.append(i)\n            if e == eidx:\n                intoken = False\n    return frozenset(token_idxs)\n\ndef convert_opinion_to_tuple(sentence):\n    text = sentence["sentence"]\n    opinions = sentence["opinions"]\n    opinion_tuples = []\n    token_offsets = list(tk.span_tokenize(text))\n    #\n    if len(opinions) > 0:\n        for opinion in opinions:\n            holder_char_idxs = opinion["Source"][1]\n            target_char_idxs = opinion["Target"][1]\n            exp_char_idxs = opinion["Polar_expression"][1]\n            polarity = opinion["Polarity"]\n            #\n            holder = convert_char_offsets_to_token_idxs(holder_char_idxs, token_offsets)\n            target = convert_char_offsets_to_token_idxs(target_char_idxs, token_offsets)\n            exp = convert_char_offsets_to_token_idxs(exp_char_idxs, token_offsets)\n            opinion_tuples.append((holder, target, exp, polarity))\n    return opinion_tuples\n\ndef parse_to_origin(sentence, label_seq):\n    if len(label_seq) == 0:\n        return []\n\n    constructed_opinions = []\n    for opinion in label_seq:\n        dict = {}\n        if len(opinion) != 4:\n            return []\n        holder, target, expression, sentiment = opinion\n\n\n        if type(holder) != str:\n            holder = \'NULL\'\n        if type(target) != str:\n            target = \'NULL\'\n        if type(expression) != str:\n            expression = \'NULL\'\n        \n            \n\n        if holder == \'NULL\':\n            dict[\'Source\'] = [[], []]\n        else:\n            begin = sentence.find(holder)\n            if begin == -1:\n                dict[\'Source\'] = [[], []]\n            else:\n                end = begin + len(holder)\n                dict[\'Source\'] = [[holder], [f"{str(begin)}:{str(end)}"]]\n\n        dict[\'Target\'] = [[], []]\n        for item in target.split(\'AND\'):\n            item = item.strip()\n            begin = sentence.find(item)\n            if begin == -1:\n                continue\n            else:\n                end = begin + len(item)\n                dict[\'Target\'][0].append(item)\n                dict[\'Target\'][1].append(f"{str(begin)}:{str(end)}")\n\n        dict[\'Polar_expression\'] = [[], []]\n        for item in expression.split(\'AND\'):\n            item = item.strip()\n            begin = sentence.find(item)\n            if begin == -1:\n                continue\n            else:\n                end = begin + len(item)\n                dict[\'Polar_expression\'][0].append(item)\n                dict[\'Polar_expression\'][1].append(f"{str(begin)}:{str(end)}")\n\n        dict[\'Polarity\'] = sentiment.capitalize()\n\n        constructed_opinions.append(dict)\n    return constructed_opinions\n\ndef weighted_score(sent_tuple1, list_of_sent_tuples):\n    best_overlap = 0\n    holder1, target1, exp1, pol1 = sent_tuple1\n    if len(holder1) == 0:\n        holder1 = frozenset(["_"])\n    if len(target1) == 0:\n        target1 = frozenset(["_"])\n    for holder2, target2, exp2, pol2 in list_of_sent_tuples:\n        if len(holder2) == 0:\n            holder2 = frozenset(["_"])\n        if len(target2) == 0:\n            target2 = frozenset(["_"])\n        if (\n            len(holder2.intersection(holder1)) > 0\n            and len(target2.intersection(target1)) > 0\n            and len(exp2.intersection(exp1)) > 0\n        ):\n            holder_overlap = len(holder2.intersection(holder1)) / len(holder1)\n            target_overlap = len(target2.intersection(target1)) / len(target1)\n            exp_overlap = len(exp2.intersection(exp1)) / len(exp1)\n            overlap = (holder_overlap + target_overlap + exp_overlap) / 3\n            if overlap > best_overlap:\n                best_overlap = overlap\n    return best_overlap\n\ndef sent_tuples_in_list(sent_tuple1, list_of_sent_tuples, keep_polarity=True):\n    holder1, target1, exp1, pol1 = sent_tuple1\n    if len(holder1) == 0:\n        holder1 = frozenset(["_"])\n    if len(target1) == 0:\n        target1 = frozenset(["_"])\n    for holder2, target2, exp2, pol2 in list_of_sent_tuples:\n        if len(holder2) == 0:\n            holder2 = frozenset(["_"])\n        if len(target2) == 0:\n            target2 = frozenset(["_"])\n        if (\n            len(holder1.intersection(holder2)) > 0\n            and len(target1.intersection(target2)) > 0\n            and len(exp1.intersection(exp2)) > 0\n        ):\n            if keep_polarity:\n                if pol1 == pol2:\n                    # print(holder1, target1, exp1, pol1)\n                    # print(holder2, target2, exp2, pol2)\n                    return True\n            else:\n                # print(holder1, target1, exp1, pol1)\n                # print(holder2, target2, exp2, pol2)\n                return True\n    return False\n\ndef tuple_precision(gold, pred, keep_polarity=True, weighted=True):\n    """\n    Weighted true positives / (true positives + false positives)\n    """\n    weighted_tp = []\n    tp = []\n    fp = []\n    #\n    for sent_idx in pred.keys():\n        ptuples = pred[sent_idx]\n        gtuples = gold[sent_idx]\n        for stuple in ptuples:\n            if sent_tuples_in_list(stuple, gtuples, keep_polarity):\n                if weighted:\n                    #sc = weighted_score(stuple, gtuples)\n                    #if sc != 1:\n                        #print(sent_idx)\n                        #print(sc)\n                        #print()\n                    weighted_tp.append(weighted_score(stuple, gtuples))\n                    tp.append(1)\n                else:\n                    weighted_tp.append(1)\n                    tp.append(1)\n            else:\n                #print(sent_idx)\n                fp.append(1)\n    #print("weighted tp: {}".format(sum(weighted_tp)))\n    #print("tp: {}".format(sum(tp)))\n    #print("fp: {}".format(sum(fp)))\n    return sum(weighted_tp) / (sum(tp) + sum(fp) + 0.0000000000000001)\n\n\ndef tuple_recall(gold, pred, keep_polarity=True, weighted=True):\n    """\n    Weighted true positives / (true positives + false negatives)\n    """\n    weighted_tp = []\n    tp = []\n    fn = []\n    #\n    assert len(gold) == len(pred)\n    #\n    for sent_idx in pred.keys():\n        ptuples = pred[sent_idx]\n        gtuples = gold[sent_idx]\n        for stuple in gtuples:\n            if sent_tuples_in_list(stuple, ptuples, keep_polarity):\n                if weighted:\n                    weighted_tp.append(weighted_score(stuple, ptuples))\n                    tp.append(1)\n                else:\n                    weighted_tp.append(1)\n                    tp.append(1)\n            else:\n                fn.append(1)\n    return sum(weighted_tp) / (sum(tp) + sum(fn) + 0.0000000000000001)\n\ndef tuple_f1(gold, pred, keep_polarity=True, weighted=True):\n    prec = tuple_precision(gold, pred, keep_polarity, weighted)\n    rec = tuple_recall(gold, pred, keep_polarity, weighted)\n    #print("prec: {}".format(prec))\n    #print("rec: {}".format(rec))\n    return prec, rec, 2 * (prec * rec) / (prec + rec + 0.00000000000000001)\n\nclass Result_ssa:\n    def __init__(self, data):\n        self.data = data\n\n    def __ge__(self, other):\n        return self.monitor >= other.monitor\n\n    def __gt__(self, other):\n        return self.monitor > other.monitor\n\n    @classmethod\n    def parse_from(cls, outputs):\n\n        data = {}\n        ID = 0\n        for example, prediction in outputs:\n\n\n            sentence = example[\'sentence\']\n\n            ssa_list_true = parse_quards_for_true(example[\'label_seq\'])\n\n            # if ssa_list_true == [(\'NULL\', \'NULL\', \'NULL\', \'NULL\')]:\n            #     print(\'all null, skip\')\n            #     continue\n\n            print(\'prediction:\', prediction)\n\n            original_prediction, ssa_list_pred = parse_quards_for_predict(prediction, sentence)[:2]\n\n            opinions = parse_to_origin(sentence, original_prediction)\n\n            # if example[\'dataset\'] == \'absa/dsunis\':\n            #     print(\'1\'*100)\n            #     print(example[\'opinions\'])\n            #     print(\'2\' * 100)\n            #     print(opinions)\n            #     print(\'3\' * 100)\n            #     print(original_prediction)\n\n\n            data[ID] = {\n                \'ID\': example.get(\'ID\', ID),\n                \'sentence\': sentence,\n                \'prompts\':example[\'prompts\'],\n                \'label_seq\':example[\'label_seq\'],\n                \'golden_label_parsed\':ssa_list_true,\n                \'prediction_parsed\': ssa_list_pred,\n                \'original_output_of_model\': original_prediction,\n                \'gold\':example,\n                \'pred\': {\n                    \'sent_id\':example[\'sent_id\'],\n                    \'opinions\':opinions,\n                    \'sentence\': example[\'sentence\']},\n                \'dataset\': example[\'dataset\']\n                \n            }\n            ID += 1\n        return cls(data)\n\n    def cal_metric(self):\n        golds = []\n        preds = []\n        for ID in self.data:\n            example = self.data[ID]\n            golds.append(example[\'gold\'])\n            preds.append(example[\'pred\'])\n\n\n        golds = dict([(s["sent_id"], convert_opinion_to_tuple(s)) for s in golds])\n\n        preds = dict([(s["sent_id"], convert_opinion_to_tuple(s)) for s in preds])\n\n        prec, rec, f1 = tuple_f1(golds, preds)\n\n        self.detailed_metrics = {\n            \'f1\': f1,\n            \'precision\':prec,\n            \'recall\':rec\n        }\n\n        self.monitor = self.detailed_metrics[\'f1\']\n\n    def save_prediction(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr, experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        file_name = os.path.join(output_dir, \'result\', f\'{experiment_name}_{dataset}_{subname}_seed:{seed}_rank:{rank}_alpha:{alpha}_target_module:{target_module}_dropout:{dropout}_lr:{lr}.json\')\n\n        print(\'save prediction to\', file_name)\n        save_json(\n            {\n                \'data\': self.data,\n                \'meta\': (model_name_or_path, subname, dataset, seed, lr, now)\n            },\n            file_name\n        )\n\n\n\n    def save_metric(self, output_dir, model_name_or_path, dataset, subname, seed, rank, alpha, target_module, dropout, lr,experiment_name):\n\n        now = datetime.now()\n        now = now.strftime("%Y-%m-%d")\n        performance_file_name = os.path.join(output_dir, \'performance\', \'performance.txt\')\n\n        print(\'save performace to\', performance_file_name)\n        append_new_line(performance_file_name, json.dumps({\n            \'experiment_name\':experiment_name,\n            \'metric\': self.detailed_metrics[\'f1\'],\n            \'dataset\': dataset,\n            \'subname\': subname,\n            \'seed\': seed,\n            \'time\': time.strftime(\'%Y-%m-%d %H_%M_%S\', time.localtime()),\n            \'model_name_or_path\': model_name_or_path,\n            \'lr\': lr,\n            \'rank\':rank,\n            \'alpha\': alpha,\n            \'target_module\': target_module,\n            \'dropout\': dropout,\n            \n        }))\n\n    def report(self):\n        for metric_names in ((\'precision\', \'recall\', \'f1\'),):\n            for metric_name in metric_names:\n                value = self.detailed_metrics[metric_name] if metric_name in self.detailed_metrics else 0\n                print(f\'{metric_name}: {value:.4f}\', end=\' | \')\n            print()\n'}
for relative, content in PATCHED_SOURCES.items():
    if USE_PRIVATE_REPO:
        break
    destination = BASE / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding='utf-8')
print('Source:', PROJECT_DIR)


## Dữ liệu benchmark
Nếu đã attach dữ liệu vào Kaggle, đặt `DATA_DIR` ở cell cấu hình để bỏ qua tải.
Mặc định chỉ tải các file trong `datasets/` từ commit cố định của repo tác giả, không cần đăng nhập GitHub.


In [ ]:
import urllib.request, zipfile
UPSTREAM_COMMIT = '2810fc2826e8f7c2ae6a0d6dd328ab7d8979744a'
if DATA_DIR is None:
    DATA_DIR = PROJECT_DIR / 'datasets'
    marker = DATA_DIR / '.source-commit'
    if not USE_PRIVATE_REPO and (not marker.exists() or marker.read_text() != UPSTREAM_COMMIT):
        archive = BASE / 'upstream-data.zip'
        url = f'https://codeload.github.com/HITSZ-HLT/Sentiment-Distillation/zip/{UPSTREAM_COMMIT}'
        urllib.request.urlretrieve(url, archive)
        with zipfile.ZipFile(archive) as source:
            for item in source.infolist():
                parts = Path(item.filename).parts
                if len(parts) < 3 or parts[1] != 'datasets' or item.is_dir():
                    continue
                destination = (PROJECT_DIR / Path(*parts[1:])).resolve()
                if not destination.is_relative_to(DATA_DIR.resolve()):
                    raise ValueError('Unsafe archive path')
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(source.read(item))
        marker.write_text(UPSTREAM_COMMIT)
        archive.unlink()
DATA_DIR = Path(DATA_DIR).resolve()
for dataset in DATASETS:
    for split in ('test', DEMO_SPLIT):
        path = DATA_DIR / dataset / f'{split}.json'
        if not path.is_file():
            raise FileNotFoundError(f'Missing {path}; change DATA_DIR or DEMO_SPLIT')
print('Dataset root:', DATA_DIR)


## Cài dependency mà không thay các gói hệ thống
Hỗ trợ cấu hình Python 3.10–3.12; phiên bản khác sẽ dừng thay vì ép cài wheel không tương thích.
Cell tạo venv, khóa phiên bản PyTorch/CUDA đã có và xem kế hoạch pip trước khi cài.
Nếu kế hoạch định thay torch, triton hoặc các gói NVIDIA, cell sẽ dừng.
Log đầy đủ và kế hoạch cài được lưu trong `output/`.


In [ ]:
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError(f'Python {sys.version_info[:2]} chưa nằm trong cấu hình dependency này (3.10–3.12).')
PROCESS_ENV = os.environ.copy()
PROCESS_ENV.pop('PYTHONPATH', None)
PROCESS_ENV.update(PYTHONNOUSERSITE='1', USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')

def run_logged(command, filename, cwd=PROJECT_DIR):
    log = OUTPUT / filename
    print('Running:', ' '.join(map(str, command[:5])), '... ->', log.name, flush=True)
    with log.open('w') as stream:
        result = subprocess.run(list(map(str, command)), cwd=cwd, env=PROCESS_ENV,
                                stdout=stream, stderr=subprocess.STDOUT)
    if result.returncode:
        print(log.read_text(errors='replace')[-8000:])
        raise RuntimeError(f'Command failed ({result.returncode}); inspect {log}')
    return log

probe = """
import torch, json, importlib.metadata as md
core = {d.metadata['Name']: d.version for d in md.distributions()
        if d.metadata['Name'].lower() in ('torch', 'triton') or d.metadata['Name'].lower().startswith('nvidia-')}
print(json.dumps(dict(torch=torch.__version__, torch_file=torch.__file__, cuda=torch.version.cuda,
                     gpu=torch.cuda.is_available(), constraints=core)))
"""
host = json.loads(subprocess.check_output([sys.executable, '-c', probe], env=PROCESS_ENV, text=True))
if not host['gpu']:
    raise RuntimeError('Chưa có CUDA GPU. Bật GPU trong Settings/Runtime rồi chạy lại.')
(OUTPUT / 'host-environment.json').write_text(json.dumps(host, indent=2))
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(ENV_DIR)], check=True, env=PROCESS_ENV)
run_logged([PYTHON, '-m', 'pip', 'install', 'pip==25.3'], 'pip-bootstrap.log')
constraints = BASE / 'host-torch-constraints.txt'
constraints.write_text('\n'.join(f'{name}=={v}' for name, v in host['constraints'].items()) + '\n')
install = [PYTHON, '-m', 'pip', 'install', '--upgrade', '--upgrade-strategy', 'only-if-needed',
           '--constraint', constraints, '-r', PROJECT_DIR / 'requirements-notebook.txt']
if FOUR_BIT:
    install.append('bitsandbytes==0.45.5')
plan = OUTPUT / 'dependency-plan.json'
run_logged(install + ['--dry-run', '--report', plan], 'dependency-resolve.log')
changes = [entry['metadata']['name'].lower().replace('_', '-') for entry in json.loads(plan.read_text())['install']]
blocked = [name for name in changes if name in ('torch', 'triton') or name.startswith('nvidia-')]
if blocked:
    raise RuntimeError(f'Kế hoạch cài định thay CUDA/PyTorch: {blocked}. Dừng để giữ môi trường nền tảng.')
print('Project packages to install:', changes)
run_logged(install, 'dependency-install.log')
inside = json.loads(subprocess.check_output([str(PYTHON), '-c', probe], env=PROCESS_ENV, text=True))
if inside['torch'] != host['torch'] or inside['torch_file'] != host['torch_file']:
    raise RuntimeError('PyTorch trong venv khác bản hệ thống; dừng để kiểm tra.')
print('Dependencies installed in venv; host torch/CUDA retained:', inside['torch'], inside['cuda'])


## Kiểm tra môi trường thực tế
Kiểm tra dependency của những gói evaluator sử dụng, thay vì yêu cầu toàn bộ ứng dụng cài sẵn trên Kaggle tương thích.
Sau đó import thư viện, tạo model nhỏ ngẫu nhiên không cần tải weights và thử forward/generate trên GPU.
Nếu bật 4-bit, thử thêm phép tính bitsandbytes trên GPU. Các kiểm tra này chưa đảm bảo model thật sẽ đủ VRAM.


In [ ]:
check = [PYTHON, PROJECT_DIR / 'check_notebook_environment.py', '--require-gpu',
         '--report', OUTPUT / 'environment-report.json']
if FOUR_BIT:
    check.append('--four-bit')
log = run_logged(check, 'environment-check.log')
print(log.read_text()[-10000:])
# Ghi lại dependency hiệu lực để chẩn đoán/tái lập môi trường.
run_logged([PYTHON, '-m', 'pip', 'freeze'], 'effective-packages.txt')
ENVIRONMENT_OK = True


## Chạy đánh giá / tiếp tục tiến độ
Mặc định chạy thử 32 mẫu SST2. Đổi `MAX_SAMPLES=0` để chạy đầy đủ một dataset.
Để chạy 12 dataset, dùng danh sách ở cell kết quả; nên chia theo dataset và seed để phù hợp thời lượng phiên.
Checkpoint lưu sau từng batch. Chạy lại cùng cấu hình sẽ dùng các dự đoán đã có; cấu hình/data/môi trường khác sẽ bị từ chối.

Giới hạn token có thể cắt bớt instruction/demo và làm thay đổi điểm so với paper; số prompt bị cắt được lưu trong metadata.
Nếu chính câu test dài hơn giới hạn, evaluator dừng để bạn tăng giới hạn thay vì âm thầm bỏ nội dung test.
Không tự đổi precision, số shot hoặc quantization khi hết VRAM vì chúng làm thay đổi điều kiện thực nghiệm.


In [ ]:
if not globals().get('ENVIRONMENT_OK'):
    raise RuntimeError('Chạy cell kiểm tra môi trường thành công trước.')
for dataset in DATASETS:
    for seed in SEEDS:
        configuration = dict(model=MODEL, dataset=dataset, seed=seed, samples=MAX_SAMPLES,
                             batch=BATCH_SIZE, shots=K_SHOT, length=MAX_INPUT_TOKENS,
                             new_tokens=MAX_NEW_TOKENS, demo=DEMO_SPLIT, dtype=DTYPE, four_bit=FOUR_BIT)
        key = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()[:12]
        slug = dataset.replace('/', '-') + f'-seed{seed}-{key}'
        script = 'fsa.py' if dataset.startswith('absa/') else 'icl_multifaced.py'
        command = [PYTHON, PROJECT_DIR / script, '--dataset', dataset, '--subname', EXPERIMENT,
                   '--seed', str(seed), '--model_name_or_path', MODEL,
                   '--data_dir', DATA_DIR, '--output_dir', OUTPUT,
                   '--eval_batch_size', str(BATCH_SIZE), '--k_shot', str(K_SHOT),
                   '--max_seq_length', str(MAX_INPUT_TOKENS), '--max_new_tokens', str(MAX_NEW_TOKENS),
                   '--max_samples', str(MAX_SAMPLES), '--demo_split', DEMO_SPLIT, '--dtype', DTYPE,
                   '--resume_file', OUTPUT / f'{slug}.checkpoint.json']
        if FOUR_BIT:
            command.append('--load_in_4bit')
        print('Evaluating', dataset, 'seed', seed, flush=True)
        log = run_logged(command, f'{slug}.log')
        print(log.read_text(errors='replace')[-2000:])


## Kết quả và lưu lại trước khi kết thúc phiên
`result.txt` chứa điểm; JSON chứa dự đoán; checkpoint dùng để tiếp tục; log/report dùng để báo lỗi.
Dataset chưa chạy được ghi N/A, không tính thành 0; số liệu chạy thử được tách khỏi đánh giá toàn bộ.
Kaggle: Save Version để lưu outputs hoặc tải file ZIP bên dưới. Phiên mới cần attach/copy lại checkpoint vào đúng `OUTPUT`.
Colab: tải ZIP hoặc đổi OUTPUT sang Google Drive đã mount. Dữ liệu trên ổ VM không được đảm bảo sau khi phiên kết thúc.
Không đặt venv/model cache trong file kết quả xuất ra.


In [ ]:
summary = subprocess.run([str(PYTHON), str(PROJECT_DIR / 'parse.py'), '--path', str(OUTPUT / 'result.txt')],
                         check=True, env=PROCESS_ENV, cwd=PROJECT_DIR, capture_output=True, text=True)
print(summary.stdout)
(OUTPUT / 'summary.txt').write_text(summary.stdout)
print('All benchmark datasets:', [
    'sc/imdb', 'sc/yelp2', 'sc/sst2', 'sc/twitter',
    'multifaced/irony18', 'multifaced/tweeteval', 'multifaced/pstance', 'multifaced/intimacy',
    'absa/asqp_rest16', 'absa/opener', 'absa/atsa_rest16', 'absa/acsa_rest16'])
archive = shutil.make_archive(str(BASE.parent / 'sentiment-evaluation-results'), 'zip', OUTPUT)
from IPython.display import FileLink, display
display(FileLink(archive))


## Giới hạn còn lại / khi gửi kết quả cho người bảo trì
- Kaggle/Colab quyết định GPU, quota, RAM và thời lượng phiên; notebook không loại bỏ các giới hạn đó.
- 4-bit chỉ là tùy chọn và có thể đổi điểm; giữ tắt nếu muốn so sánh precision gốc.
- Chạy một GPU, model 1B–1.5B trước; batch 1 vẫn có thể hết VRAM nếu prompt dài.
- Nếu hết VRAM: giảm batch nếu >1; thử 4-bit hoặc giảm shot/độ dài và đặt tên thực nghiệm khác.
- Nếu lỗi dependency, gửi `host-environment.json`, `dependency-resolve.log`, `dependency-install.log`,
  `environment-check.log`, `environment-report.json` (nếu có) và log dataset. Không gửi token.
- Muốn đối chiếu paper: dùng cùng split, prompt, precision và trung bình seed 42/52/62. Source hiện lấy demo
  từ train, còn paper mô tả dev; một số dataset local không có val.json nên cần chuẩn bị split tương ứng.

Nguồn: [Kaggle Notebook docs](https://www.kaggle.com/docs/notebooks),
[Colab FAQ](https://research.google.com/colaboratory/faq.html),
[Paper](https://arxiv.org/html/2503.03225v2),
[Repo tác giả](https://github.com/HITSZ-HLT/Sentiment-Distillation).
